[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/ch08_02_lstm_from_scratch.ipynb)

# Chapter 8-2: LSTM(장단기 기억 네트워크) 밑바닥부터 구현하기

> **목표**: LSTM의 수학적 원리를 **논문 수준**으로 완전히 이해하고, NumPy로 직접 구현한 뒤, PyTorch와 비교합니다.
>
> **대상**: 딥러닝을 처음 공부하는 분 (RNN 기초를 먼저 학습하면 좋습니다 - ch08_01 참고)
>
> **소요 시간**: 약 3~4시간
>
> **참고 논문**: Hochreiter & Schmidhuber, "Long Short-Term Memory" (1997)

---

## 📋 목차

| 파트 | 내용 | 핵심 |
|------|------|------|
| **Part 1** | LSTM 이론 | RNN 한계 복습, Cell State, 3개 게이트 수식 완전 분해, 역전파 |
| **Part 2** | NumPy 밑바닥 구현 | LSTMCell 클래스 (forward/backward), 게이트 값 시각화 |
| **Part 3** | PyTorch nn.LSTM | 감정 분류 예제, hidden/cell state 차이 |
| **Part 4** | 정리 | LSTM 한계, Attention 동기, 핵심 수식 요약표 |

In [ ]:
# ============================================================
# Colab 환경 설정 셀
# Google Colab에서 실행할 때 필요한 패키지를 설치합니다.
# 로컬 환경에서는 이미 설치되어 있다면 건너뛰어도 됩니다.
# ============================================================

# numpy: 행렬 연산을 위한 핵심 라이브러리 (LSTM 밑바닥 구현에 사용)
# matplotlib: 게이트 값, 기울기 흐름 등을 시각화하기 위한 그래프 라이브러리
# torch: PyTorch - Part 3에서 nn.LSTM과 비교할 때 사용
# torchtext: 감정 분류 예제에서 텍스트 데이터 처리에 사용
!pip install -q numpy matplotlib torch

# 설치 확인을 위한 버전 출력
import numpy as np  # NumPy를 np라는 별칭으로 불러옴
import matplotlib.pyplot as plt  # 시각화 라이브러리를 plt로 불러옴
import torch  # PyTorch를 torch로 불러옴
import torch.nn as nn  # PyTorch의 신경망 모듈을 nn으로 불러옴

print(f"NumPy 버전: {np.__version__}")  # NumPy 버전 확인
print(f"PyTorch 버전: {torch.__version__}")  # PyTorch 버전 확인

# 한글 폰트 설정 (시각화에서 한글이 깨지지 않도록)
plt.rcParams['font.family'] = 'AppleGothic'  # macOS용 한글 폰트
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

# Colab에서는 아래 폰트를 사용 (위 설정이 안 되면 아래로 대체)
try:
    import matplotlib.font_manager as fm  # 폰트 매니저 불러오기
    # Colab에는 NanumGothic이 기본 설치되어 있음
    plt.rcParams['font.family'] = 'NanumGothic'  # Colab용 한글 폰트
except Exception:
    pass  # 로컬 환경에서는 AppleGothic 유지

# 재현성을 위한 랜덤 시드 고정
np.random.seed(42)  # NumPy 랜덤 시드 고정
torch.manual_seed(42)  # PyTorch 랜덤 시드 고정

---

# Part 1: LSTM 이론 (수식 완전 분해)

---

## 1.1 RNN의 한계 복습: 기울기 소실 문제

이전 챕터(ch08_01)에서 배운 RNN의 핵심 수식을 떠올려 봅시다:

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$

여기서:
- $h_t$: 시점 $t$의 은닉 상태 (hidden state)
- $h_{t-1}$: 이전 시점의 은닉 상태
- $x_t$: 시점 $t$의 입력
- $W_{hh}$, $W_{xh}$: 가중치 행렬
- $b_h$: 편향(bias)

### 왜 기울기가 소실될까?

역전파(Backpropagation Through Time, BPTT)에서 시점 $t$에서 시점 $k$까지의 기울기는 다음과 같이 **연쇄 법칙(chain rule)**으로 계산됩니다:

$$\frac{\partial h_t}{\partial h_k} = \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} = \prod_{i=k+1}^{t} W_{hh}^T \cdot \text{diag}(1 - h_i^2)$$

여기서 핵심 문제:
1. **tanh의 미분값**: $1 - h_i^2$는 항상 **0~1 사이**입니다
2. **행렬의 반복 곱**: $W_{hh}$를 계속 곱하면, 고유값(eigenvalue)이 1보다 작으면 **기울기가 0으로 수렴** (소실)
3. **결과**: 시퀀스가 길어질수록 (t - k가 클수록) 과거 정보의 기울기가 **지수적으로 감소**

> 💡 **비유**: RNN은 "전화기 게임"과 같습니다. 앞 사람이 한 말을 계속 전달하면 원래 메시지가 점점 왜곡되듯이, 먼 과거의 정보가 현재까지 제대로 전달되지 않습니다.

In [ ]:
# ============================================================
# RNN의 기울기 소실을 시각적으로 확인하는 코드
# ============================================================

# 시퀀스 길이를 설정합니다 (시점의 개수)
seq_lengths = np.arange(1, 51)  # 1부터 50까지의 시퀀스 길이

# 세 가지 경우를 비교합니다
# case 1: tanh 미분값이 0.9인 경우 (비교적 큰 값)
gradient_09 = 0.9 ** seq_lengths  # 0.9를 시퀀스 길이만큼 거듭제곱

# case 2: tanh 미분값이 0.5인 경우 (중간 값)
gradient_05 = 0.5 ** seq_lengths  # 0.5를 시퀀스 길이만큼 거듭제곱

# case 3: tanh 미분값이 0.25인 경우 (작은 값)
gradient_025 = 0.25 ** seq_lengths  # 0.25를 시퀀스 길이만큼 거듭제곱

# 그래프를 그립니다
fig, ax = plt.subplots(1, 1, figsize=(10, 5))  # 10x5 크기의 그래프 생성

# 각 경우의 기울기를 선 그래프로 표시
ax.plot(seq_lengths, gradient_09, label='기울기 계수 = 0.9', linewidth=2)  # 0.9 케이스
ax.plot(seq_lengths, gradient_05, label='기울기 계수 = 0.5', linewidth=2)  # 0.5 케이스
ax.plot(seq_lengths, gradient_025, label='기울기 계수 = 0.25', linewidth=2)  # 0.25 케이스

ax.set_xlabel('시퀀스 길이 (시점 간 거리)', fontsize=12)  # x축 레이블
ax.set_ylabel('기울기 크기 (상대값)', fontsize=12)  # y축 레이블
ax.set_title('RNN의 기울기 소실: 시퀀스가 길어질수록 기울기가 급격히 감소', fontsize=14)  # 제목
ax.legend(fontsize=11)  # 범례 표시
ax.set_yscale('log')  # y축을 로그 스케일로 (지수적 감소를 보기 위해)
ax.grid(True, alpha=0.3)  # 격자 표시 (투명도 0.3)
ax.axhline(y=1e-10, color='red', linestyle='--', alpha=0.5, label='사실상 0')  # 기울기가 거의 0인 기준선

plt.tight_layout()  # 그래프 레이아웃 자동 조정
plt.show()  # 그래프 출력

print("\n📊 관찰: 기울기 계수가 1보다 작으면, 시퀀스가 길어질수록 기울기가 0에 수렴합니다.")
print("이것이 RNN이 긴 시퀀스를 학습하지 못하는 근본적인 이유입니다.")

---

## 1.2 LSTM의 핵심 아이디어: Cell State (셀 상태)

### LSTM은 어떻게 기울기 소실을 해결할까?

LSTM(Long Short-Term Memory)은 1997년 Hochreiter & Schmidhuber가 제안한 구조로,
**Cell State(셀 상태)**라는 별도의 정보 경로를 도입합니다.

### Cell State = "컨베이어 벨트" 비유

```
시점 1          시점 2          시점 3          시점 4
  |               |               |               |
  v               v               v               v
──[c₁]─────────[c₂]─────────[c₃]─────────[c₄]──── (Cell State: 컨베이어 벨트)
  |    약간 수정    |    약간 수정    |    약간 수정    |
  v               v               v               v
──[h₁]─────────[h₂]─────────[h₃]─────────[h₄]──── (Hidden State: 출력)
```

**핵심 차이점:**

| | RNN | LSTM |
|---|---|---|
| 정보 전달 | $h_t = \tanh(W \cdot h_{t-1} + ...)$ | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ |
| 연산 | **행렬 곱 → tanh** (비선형 압축) | **원소별 곱 + 덧셈** (선형에 가까움) |
| 기울기 | 매 시점 **비선형 변환** 통과 → 소실 | **덧셈 구조** → 기울기가 직접 흐름 |

> 💡 **비유**: RNN이 "전화기 게임"이었다면, LSTM의 Cell State는 **"메모장"**입니다.
> - 컨베이어 벨트 위의 메모장에 정보가 적혀 있고
> - 각 시점에서 **지울 내용을 지우고** (Forget Gate)
> - **새로 적을 내용을 적고** (Input Gate)
> - **필요한 내용을 읽어서** 출력합니다 (Output Gate)

### LSTM의 구성 요소

LSTM 셀에는 **4가지 핵심 요소**가 있습니다:

1. **Cell State ($c_t$)**: 장기 기억을 저장하는 컨베이어 벨트
2. **Forget Gate ($f_t$)**: 이전 기억 중 **얼마나 잊을지** 결정
3. **Input Gate ($i_t$)**: 새로운 정보를 **얼마나 기억할지** 결정
4. **Output Gate ($o_t$)**: Cell State에서 **얼마나 출력할지** 결정

---

## 1.3 LSTM 게이트 수식 완전 분해

이제 LSTM의 수식을 **한 줄 한 줄** 분해합니다. 먼저 전체 표기법을 정리합니다.

### 표기법 (Notation)

| 기호 | 의미 | 차원 (크기) |
|------|------|-------------|
| $x_t$ | 시점 $t$의 입력 벡터 | $(d,)$ — $d$는 입력 차원 |
| $h_{t-1}$ | 이전 시점의 은닉 상태 | $(H,)$ — $H$는 은닉 차원 |
| $c_{t-1}$ | 이전 시점의 셀 상태 | $(H,)$ — 은닉 차원과 동일 |
| $[h_{t-1}, x_t]$ | $h_{t-1}$과 $x_t$를 연결(concatenate) | $(H + d,)$ |
| $W_f, W_i, W_c, W_o$ | 각 게이트의 가중치 행렬 | $(H, H + d)$ |
| $b_f, b_i, b_c, b_o$ | 각 게이트의 편향 벡터 | $(H,)$ |
| $\sigma$ | 시그모이드 함수: $\sigma(x) = \frac{1}{1 + e^{-x}}$ | 출력이 $(0, 1)$ 사이 |
| $\tanh$ | 하이퍼볼릭 탄젠트: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | 출력이 $(-1, 1)$ 사이 |
| $\odot$ | 원소별 곱 (Hadamard product) | 같은 크기 벡터끼리 |

> 💡 **차원에 대한 직관**: 만약 입력 단어를 100차원 임베딩으로 표현하고 ($d=100$),
> 은닉 상태를 256차원으로 설정하면 ($H=256$), 연결 벡터 $[h_{t-1}, x_t]$는 356차원이 됩니다.

---

### 1.3.1 Forget Gate (망각 게이트): "무엇을 잊을까?"

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**수식을 한 단계씩 분해:**

**Step 1**: 이전 은닉 상태 $h_{t-1}$과 현재 입력 $x_t$를 연결(concatenate)합니다.
$$z = [h_{t-1}, x_t] \quad \text{차원: } (H + d,)$$

**Step 2**: 가중치 행렬 $W_f$와 행렬곱을 수행합니다.
$$W_f \cdot z + b_f \quad \text{차원: } (H, H+d) \times (H+d,) + (H,) = (H,)$$

**Step 3**: 시그모이드 함수를 적용합니다.
$$f_t = \sigma(W_f \cdot z + b_f) \quad \text{출력: 각 원소가 0~1 사이}$$

**시그모이드의 역할:**
- 시그모이드 출력이 **0에 가까우면** → 해당 정보를 **완전히 잊음**
- 시그모이드 출력이 **1에 가까우면** → 해당 정보를 **완전히 기억**
- 즉, $f_t$는 이전 셀 상태 $c_{t-1}$의 각 원소에 대해 "이 정보를 몇 %나 유지할까?"를 결정

> 💡 **예시**: "오늘 날씨가 좋다"를 읽은 후 "하지만 내일은 비가 온다"를 만났을 때,
> Forget Gate는 "날씨가 좋다"라는 기존 정보의 중요도를 낮추는 역할을 합니다.

---

### 1.3.2 Input Gate (입력 게이트): "무엇을 새로 기억할까?"

Input Gate는 **두 부분**으로 구성됩니다:

**① 입력 게이트 값 (얼마나 기억할지):**
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(0~1 사이 값)}$$

**② 후보 기억 (무엇을 기억할지):**
$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) \quad \text{(-1~1 사이 값)}$$

**두 부분의 관계:**
- $\tilde{c}_t$ (후보 기억): "이번 시점에서 **새로 만든 정보**" — tanh로 -1~1 사이 값
- $i_t$ (입력 게이트): "그 새 정보를 **얼마나 반영할지**" — 시그모이드로 0~1 사이 값
- 최종적으로 $i_t \odot \tilde{c}_t$: 새 정보를 **선택적으로** 반영

> 💡 **비유**: 수업을 듣고 있다고 상상해보세요.
> - $\tilde{c}_t$는 선생님이 말씀하신 **내용 전체**입니다
> - $i_t$는 그 중에서 **내가 중요하다고 판단한 부분의 비율**입니다
> - $i_t \odot \tilde{c}_t$는 **실제로 노트에 적은 내용**입니다

---

### 1.3.3 Cell State 업데이트: "기억 갱신"

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

이 수식이 **LSTM의 가장 핵심**입니다! 한 항씩 살펴봅시다:

**항 1: $f_t \odot c_{t-1}$ (기존 기억에서 일부를 잊음)**
- $c_{t-1}$: 이전 시점까지의 셀 상태 (지금까지의 기억)
- $f_t$: 망각 게이트 (각 원소가 0~1)
- 원소별 곱 $\odot$: $c_{t-1}$의 각 원소에 $f_t$의 대응 원소를 곱함
- $f_t$가 0이면 → 해당 기억 완전 삭제
- $f_t$가 1이면 → 해당 기억 완전 유지

**항 2: $i_t \odot \tilde{c}_t$ (새로운 정보를 추가)**
- $\tilde{c}_t$: 후보 기억 (-1~1 사이)
- $i_t$: 입력 게이트 (0~1 사이)
- 원소별 곱: 새 정보를 선택적으로 필터링

**두 항의 덧셈 (+)이 핵심인 이유:**
- RNN: $h_t = \tanh(W \cdot h_{t-1} + ...)$ → **비선형 변환**을 통과
- LSTM: $c_t = f_t \odot c_{t-1} + ...$ → **덧셈**으로 연결
- 역전파 시 덧셈의 기울기는 **그대로 1**로 전달됨!
- 이것이 바로 **기울기 고속도로(gradient highway)**

> 💡 **수학적 이유**: $\frac{\partial c_t}{\partial c_{t-1}} = f_t$ 이므로,
> $f_t$가 1에 가까우면 기울기가 **거의 손실 없이** 과거로 전달됩니다.
> RNN처럼 $W^T \cdot \text{diag}(1 - h^2)$를 반복 곱하는 것보다 훨씬 안정적!

---

### 1.3.4 Output Gate (출력 게이트): "무엇을 출력할까?"

$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(c_t)$$

**수식 분해:**

**① 출력 게이트 값:**
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad \text{(0~1 사이 값)}$$
- "셀 상태의 어떤 부분을 밖으로 내보낼지" 결정

**② 은닉 상태 계산:**
$$h_t = o_t \odot \tanh(c_t)$$
- $\tanh(c_t)$: 셀 상태를 -1~1 범위로 정규화
- $o_t$: 그 중에서 출력할 비율을 결정
- $h_t$: 최종 출력 (다음 시점으로 전달 + 현재 시점의 예측에 사용)

> 💡 **Cell State vs Hidden State:**
> - **Cell State ($c_t$)**: 내부 기억. LSTM 셀 **안에서만** 순환
> - **Hidden State ($h_t$)**: 외부 출력. 다음 레이어나 최종 예측에 사용
> - 비유: $c_t$는 "머릿속 생각", $h_t$는 "실제로 말한 것"

---

### 1.3.5 LSTM 전체 수식 요약

하나의 LSTM 시점에서 일어나는 **전체 연산**을 순서대로 정리합니다:

$$\boxed{\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) & \text{(1) 망각 게이트} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) & \text{(2) 입력 게이트} \\
\tilde{c}_t &= \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) & \text{(3) 후보 기억} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t & \text{(4) 셀 상태 업데이트} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) & \text{(5) 출력 게이트} \\
h_t &= o_t \odot \tanh(c_t) & \text{(6) 은닉 상태 (출력)}
\end{aligned}}$$

In [ ]:
# ============================================================
# 시그모이드(σ)와 tanh 함수를 시각적으로 비교합니다
# LSTM의 게이트에서 각각의 역할을 이해하기 위해 중요합니다
# ============================================================

# x축 값을 -6에서 6까지 생성합니다 (충분한 범위)
x_vals = np.linspace(-6, 6, 200)  # -6~6 사이 200개의 점

# 시그모이드 함수: σ(x) = 1 / (1 + e^(-x))
def sigmoid(x):
    """시그모이드 함수 - 출력이 0~1 사이"""
    return 1 / (1 + np.exp(-x))  # 시그모이드 수식 그대로 구현

# tanh 함수: 이미 numpy에 내장되어 있음
# tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))

# 2개의 서브플롯을 나란히 배치합니다
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1행 2열, 14x5 크기

# --- 왼쪽: 시그모이드 ---
axes[0].plot(x_vals, sigmoid(x_vals), color='blue', linewidth=2)  # 시그모이드 그래프
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)  # y=0.5 기준선
axes[0].axhline(y=0, color='gray', linestyle='-', alpha=0.3)  # y=0 기준선
axes[0].axhline(y=1, color='gray', linestyle='-', alpha=0.3)  # y=1 기준선
axes[0].axvline(x=0, color='gray', linestyle='-', alpha=0.3)  # x=0 기준선
axes[0].set_title('시그모이드 σ(x): 게이트용 (0~1)', fontsize=14)  # 제목
axes[0].set_xlabel('x', fontsize=12)  # x축 레이블
axes[0].set_ylabel('σ(x)', fontsize=12)  # y축 레이블
axes[0].set_ylim(-0.1, 1.1)  # y축 범위 설정
axes[0].grid(True, alpha=0.3)  # 격자 표시
# 주요 포인트에 화살표 주석 추가
axes[0].annotate('큰 양수 → 1 (기억)', xy=(4, sigmoid(4)), fontsize=10,  # 화살표 주석
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(2, 0.7))
axes[0].annotate('큰 음수 → 0 (망각)', xy=(-4, sigmoid(-4)), fontsize=10,  # 화살표 주석
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(-3, 0.3))

# --- 오른쪽: tanh ---
axes[1].plot(x_vals, np.tanh(x_vals), color='orange', linewidth=2)  # tanh 그래프
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)  # y=0 기준선
axes[1].axhline(y=-1, color='gray', linestyle='--', alpha=0.3)  # y=-1 기준선
axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.3)  # y=1 기준선
axes[1].axvline(x=0, color='gray', linestyle='-', alpha=0.3)  # x=0 기준선
axes[1].set_title('tanh(x): 값/출력용 (-1~1)', fontsize=14)  # 제목
axes[1].set_xlabel('x', fontsize=12)  # x축 레이블
axes[1].set_ylabel('tanh(x)', fontsize=12)  # y축 레이블
axes[1].set_ylim(-1.3, 1.3)  # y축 범위 설정
axes[1].grid(True, alpha=0.3)  # 격자 표시
axes[1].annotate('양수/음수 모두 표현 가능', xy=(3, np.tanh(3)), fontsize=10,  # 주석
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(0.5, 0.5))

plt.tight_layout()  # 레이아웃 자동 조정
plt.show()  # 그래프 출력

print("\n🔑 핵심 정리:")
print("  시그모이드 σ: 0~1 사이 → '게이트' 역할 (얼마나 통과시킬지)")
print("  tanh:        -1~1 사이 → '값' 역할 (어떤 정보를 만들지/출력할지)")

---

## 1.4 LSTM에서의 역전파: Cell State를 통한 기울기 고속도로

### RNN vs LSTM의 기울기 흐름 비교

**RNN의 기울기 흐름:**

시점 $t$에서 시점 $t-k$까지의 기울기:
$$\frac{\partial L}{\partial h_{t-k}} = \frac{\partial L}{\partial h_t} \cdot \prod_{i=t-k+1}^{t} \underbrace{W_{hh}^T \cdot \text{diag}(1 - h_i^2)}_{\text{매 시점 행렬곱 + 비선형 변환}}$$

문제: 이 곱이 **지수적으로 감소**하거나 **폭발**합니다.

**LSTM의 기울기 흐름 (Cell State 경로):**

셀 상태 $c_t$에 대한 기울기:
$$\frac{\partial c_t}{\partial c_{t-1}} = f_t$$

따라서 시점 $t$에서 $t-k$까지:
$$\frac{\partial c_t}{\partial c_{t-k}} = \prod_{i=t-k+1}^{t} f_i$$

핵심 차이점:
1. **곱하는 값**: RNN은 $W_{hh}^T \cdot \text{diag}(...)$ → LSTM은 $f_t$ (스칼라에 가까움)
2. **범위**: $f_t$는 시그모이드 출력이므로 **0~1 사이**
3. **학습 가능**: $f_t$는 **학습으로 조절**됨 → 중요한 정보일 때 $f_t \approx 1$로 기울기 보존

> 💡 **"기울기 고속도로"**: Cell State를 통한 경로는 마치 **고속도로**와 같습니다.
> RNN의 기울기가 구불구불한 시골길(매번 비선형 변환)을 지나는 반면,
> LSTM의 기울기는 고속도로(덧셈 + 원소별 곱)를 통해 빠르게 전달됩니다.

### LSTM의 완전한 역전파 수식

손실 $L$에서 각 파라미터까지의 기울기를 구하려면, 먼저 각 게이트에 대한 기울기를 계산합니다:

**1단계**: 출력에서 셀 상태로의 기울기
$$\delta_{h_t} = \frac{\partial L}{\partial h_t}$$
$$\delta_{c_t} = \delta_{h_t} \odot o_t \odot (1 - \tanh^2(c_t)) + \delta_{c_{t+1}} \odot f_{t+1}$$

여기서 $\delta_{c_{t+1}} \odot f_{t+1}$이 바로 **기울기 고속도로**입니다!

**2단계**: 각 게이트로의 기울기
$$\delta_{f_t} = \delta_{c_t} \odot c_{t-1} \odot f_t \odot (1 - f_t) \quad \text{(시그모이드 미분)}$$
$$\delta_{i_t} = \delta_{c_t} \odot \tilde{c}_t \odot i_t \odot (1 - i_t)$$
$$\delta_{\tilde{c}_t} = \delta_{c_t} \odot i_t \odot (1 - \tilde{c}_t^2) \quad \text{(tanh 미분)}$$
$$\delta_{o_t} = \delta_{h_t} \odot \tanh(c_t) \odot o_t \odot (1 - o_t)$$

**3단계**: 가중치에 대한 기울기
$$\frac{\partial L}{\partial W_f} = \delta_{f_t} \cdot [h_{t-1}, x_t]^T \quad \text{(외적)}$$
(나머지 $W_i, W_c, W_o$도 동일한 패턴)

In [ ]:
# ============================================================
# RNN vs LSTM 기울기 흐름 비교 다이어그램
# 텍스트 기반 ASCII 아트로 두 구조를 비교합니다
# ============================================================

# RNN과 LSTM의 기울기 흐름을 시각적으로 비교하는 다이어그램
fig, axes = plt.subplots(2, 1, figsize=(14, 10))  # 2행 1열, 14x10 크기

# --- 상단: RNN 기울기 흐름 ---
ax1 = axes[0]  # 첫 번째 서브플롯
ax1.set_xlim(0, 10)  # x축 범위
ax1.set_ylim(0, 4)  # y축 범위
ax1.set_title('RNN 기울기 흐름: 매 시점 행렬곱 + tanh → 기울기 소실', fontsize=14)  # 제목

# RNN 셀들을 그립니다
for i, t in enumerate([1, 3, 5, 7]):  # 4개 시점
    # 셀 사각형 그리기
    rect = plt.Rectangle((t-0.4, 1.5), 0.8, 1.0, fill=True,  # 사각형 생성
                         facecolor='lightcoral', edgecolor='black', linewidth=2)
    ax1.add_patch(rect)  # 사각형 추가
    ax1.text(t, 2.0, f'tanh', ha='center', va='center', fontsize=10, fontweight='bold')  # 셀 레이블
    ax1.text(t, 1.2, f'h_{i+1}', ha='center', fontsize=10)  # 은닉 상태 레이블

# 기울기 흐름 화살표 (점점 얇아짐 = 기울기 소실)
widths = [3.0, 2.0, 1.0, 0.3]  # 화살표 너비 (점점 줄어듦)
colors = ['red', 'orangered', 'orange', 'lightyellow']  # 색상 (점점 연해짐)
for i, (t1, t2) in enumerate([(7, 5), (5, 3), (3, 1)]):
    ax1.annotate('', xy=(t1-0.4, 2.0), xytext=(t2+0.4, 2.0),  # 화살표
                arrowprops=dict(arrowstyle='->', color=colors[i],
                              linewidth=widths[i], mutation_scale=15))
    # 기울기 크기 표시
    gradient_val = 0.5 ** (i + 1)  # 기울기가 절반씩 줄어든다고 가정
    ax1.text((t1+t2)/2, 2.7, f'기울기: {gradient_val:.3f}', ha='center', fontsize=9,  # 기울기 값
            color=colors[i], fontweight='bold')

ax1.text(9, 2.0, '← 기울기 역전파 방향', fontsize=10, va='center')  # 방향 레이블
ax1.axis('off')  # 축 숨김

# --- 하단: LSTM 기울기 흐름 ---
ax2 = axes[1]  # 두 번째 서브플롯
ax2.set_xlim(0, 10)  # x축 범위
ax2.set_ylim(0, 5)  # y축 범위
ax2.set_title('LSTM 기울기 흐름: Cell State를 통한 기울기 고속도로', fontsize=14)  # 제목

# Cell State 라인 (고속도로)
ax2.plot([0.5, 8.5], [4.0, 4.0], color='green', linewidth=4, linestyle='-')  # 셀 상태 경로
ax2.text(4.5, 4.3, 'Cell State (c): 기울기 고속도로 ← 덧셈으로 연결!', ha='center',  # 레이블
        fontsize=11, color='green', fontweight='bold')

# Hidden State 라인
ax2.plot([0.5, 8.5], [1.5, 1.5], color='blue', linewidth=2, linestyle='--')  # 은닉 상태 경로
ax2.text(4.5, 1.1, 'Hidden State (h): 출력 경로', ha='center', fontsize=10, color='blue')  # 레이블

# LSTM 셀들
for i, t in enumerate([1, 3, 5, 7]):  # 4개 시점
    # LSTM 셀 사각형
    rect = plt.Rectangle((t-0.4, 2.2), 0.8, 1.5, fill=True,  # 사각형 생성
                         facecolor='lightgreen', edgecolor='black', linewidth=2)
    ax2.add_patch(rect)  # 사각형 추가
    ax2.text(t, 3.2, f'LSTM', ha='center', va='center', fontsize=9, fontweight='bold')  # 셀 레이블
    ax2.text(t, 2.8, f'Cell {i+1}', ha='center', fontsize=8)  # 셀 번호
    # Cell State 연결 (굵은 초록색 = 기울기 잘 전달)
    ax2.plot([t, t], [3.7, 4.0], color='green', linewidth=2)  # 셀→셀상태 연결선
    # Hidden State 연결
    ax2.plot([t, t], [1.5, 2.2], color='blue', linewidth=1.5, linestyle='--')  # 셀→은닉상태 연결선

# Cell State 기울기 화살표 (거의 동일한 크기 유지)
widths_lstm = [3.0, 2.8, 2.5]  # 화살표 너비 (거의 동일!)
for i, (t1, t2) in enumerate([(7, 5), (5, 3), (3, 1)]):
    ax2.annotate('', xy=(t1-0.4, 4.0), xytext=(t2+0.4, 4.0),  # 화살표
                arrowprops=dict(arrowstyle='->', color='green',
                              linewidth=widths_lstm[i], mutation_scale=15))
    gradient_val = 0.95 ** (i + 1)  # f_t ≈ 0.95일 때 기울기가 잘 보존됨
    ax2.text((t1+t2)/2, 4.5, f'기울기: {gradient_val:.3f}', ha='center', fontsize=9,  # 기울기 값
            color='darkgreen', fontweight='bold')

ax2.text(9, 4.0, '← 기울기', fontsize=10, va='center', color='green')  # 방향 레이블
ax2.axis('off')  # 축 숨김

plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print("\n🔑 핵심 비교:")
print("  RNN: 기울기가 0.500 → 0.250 → 0.125 (급격히 감소)")
print("  LSTM: 기울기가 0.950 → 0.903 → 0.857 (천천히 감소)")
print("  → LSTM의 Cell State 덧셈 구조 덕분에 먼 과거 정보도 학습 가능!")

---

## 1.5 LSTM 변형: Peephole Connection과 GRU

### Peephole Connection (엿보기 연결)

기본 LSTM의 게이트들은 $h_{t-1}$과 $x_t$만 봅니다. Peephole LSTM은 **셀 상태 $c_{t-1}$도 직접 참조**합니다:

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + \underbrace{p_f \odot c_{t-1}}_{\text{peephole}} + b_f)$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + \underbrace{p_i \odot c_{t-1}}_{\text{peephole}} + b_i)$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + \underbrace{p_o \odot c_t}_{\text{peephole (주의: c_t)}} + b_o)$$

- $p_f, p_i, p_o$: peephole 가중치 벡터 (행렬이 아닌 벡터!)
- 효과: 게이트가 셀 상태의 현재 값을 직접 참고하여 더 정밀한 제어 가능
- 실제로는 성능 향상이 미미하여 잘 사용하지 않음

### GRU (Gated Recurrent Unit) — Cho et al., 2014

GRU는 LSTM을 **단순화**한 변형입니다:

$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t]) \quad \text{(Update Gate = Forget + Input 통합)}$$
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t]) \quad \text{(Reset Gate)}$$
$$\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t]) \quad \text{(후보 은닉 상태)}$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{(은닉 상태 업데이트)}$$

**LSTM vs GRU 비교:**

| 특성 | LSTM | GRU |
|------|------|-----|
| 게이트 수 | 3개 (Forget, Input, Output) | 2개 (Update, Reset) |
| 상태 | Cell State + Hidden State | Hidden State만 |
| 파라미터 수 | $4 \times H \times (H + d) + 4H$ | $3 \times H \times (H + d) + 3H$ |
| 성능 | 긴 시퀀스에서 약간 우위 | 짧은 시퀀스에서 비슷하거나 더 좋음 |
| 학습 속도 | 느림 (파라미터 많음) | 빠름 (파라미터 적음) |

> 💡 **실전 팁**: 최근에는 Transformer가 대부분의 시퀀스 작업에서 LSTM/GRU를 대체했지만,
> 실시간 시계열 처리, 에지 디바이스 등에서는 여전히 LSTM/GRU가 사용됩니다.

---

# Part 2: LSTM 밑바닥 구현 (NumPy only)

> 이제 Part 1에서 배운 수식을 **한 줄 한 줄** NumPy 코드로 구현합니다.
> 모든 코드 라인에 한글 주석으로 "이 줄이 어느 게이트의 어느 수식인지" 설명합니다.

---

In [ ]:
# ============================================================
# Part 2.1: 활성화 함수 직접 구현
# LSTM에서 사용하는 시그모이드와 tanh를 NumPy로 구현합니다.
# ============================================================

def sigmoid(x):
    """시그모이드 활성화 함수
    
    수식: σ(x) = 1 / (1 + e^(-x))
    출력 범위: (0, 1)
    용도: 게이트 값 계산 (Forget, Input, Output Gate)
    
    Args:
        x: 입력 배열 (numpy array)
    Returns:
        0~1 사이의 값으로 변환된 배열
    """
    # 수치 안정성을 위해 클리핑 적용 (너무 크거나 작은 값 방지)
    x_clipped = np.clip(x, -500, 500)  # overflow 방지를 위해 범위 제한
    return 1.0 / (1.0 + np.exp(-x_clipped))  # 시그모이드 공식 적용


def sigmoid_derivative(sigmoid_output):
    """시그모이드 함수의 미분값 계산
    
    수식: σ'(x) = σ(x) * (1 - σ(x))
    주의: 입력은 σ(x)의 '출력값'입니다 (원래 x가 아님!)
    
    Args:
        sigmoid_output: 시그모이드 함수의 출력값
    Returns:
        미분값
    """
    return sigmoid_output * (1.0 - sigmoid_output)  # σ(x) * (1 - σ(x))


def tanh_derivative(tanh_output):
    """tanh 함수의 미분값 계산
    
    수식: tanh'(x) = 1 - tanh²(x)
    주의: 입력은 tanh(x)의 '출력값'입니다 (원래 x가 아님!)
    
    Args:
        tanh_output: tanh 함수의 출력값
    Returns:
        미분값
    """
    return 1.0 - tanh_output ** 2  # 1 - tanh²(x)


# 함수 동작 확인 테스트
print("=== 활성화 함수 테스트 ===")
test_val = np.array([0.0, 1.0, -1.0, 5.0, -5.0])  # 테스트 입력값
print(f"입력값:         {test_val}")  # 입력 확인
print(f"sigmoid 출력:   {sigmoid(test_val).round(4)}")  # 시그모이드 출력 확인
print(f"tanh 출력:      {np.tanh(test_val).round(4)}")  # tanh 출력 확인
print(f"sigmoid 미분:   {sigmoid_derivative(sigmoid(test_val)).round(4)}")  # 시그모이드 미분 확인
print(f"tanh 미분:      {tanh_derivative(np.tanh(test_val)).round(4)}")  # tanh 미분 확인

In [ ]:
# ============================================================
# Part 2.2: LSTMCell 클래스 구현
# LSTM의 하나의 셀을 완전히 구현합니다.
# forward(순전파)와 backward(역전파) 메서드를 포함합니다.
# ============================================================

class LSTMCell:
    """LSTM 셀 하나를 밑바닥부터 구현한 클래스
    
    하나의 시점(time step)에서의 LSTM 연산을 수행합니다.
    forward()로 순전파, backward()로 역전파를 계산합니다.
    
    Args:
        input_size (int): 입력 벡터 x_t의 차원 (d)
        hidden_size (int): 은닉 상태 h_t의 차원 (H)
    """
    
    def __init__(self, input_size, hidden_size):
        """LSTMCell 초기화: 가중치와 편향을 생성합니다"""
        self.input_size = input_size   # 입력 차원 (d)
        self.hidden_size = hidden_size  # 은닉 차원 (H)
        
        # --------------------------------------------------
        # 가중치 초기화 (Xavier/Glorot 초기화 사용)
        # Xavier 초기화: 입출력 크기에 맞춰 적절한 분산으로 초기화
        # 이렇게 하면 학습 초기에 기울기가 너무 크거나 작지 않음
        # --------------------------------------------------
        scale = np.sqrt(2.0 / (input_size + hidden_size))  # Xavier 초기화 스케일
        
        # Forget Gate 가중치: W_f ∈ R^(H, H+d), b_f ∈ R^(H,)
        self.W_f = np.random.randn(hidden_size, hidden_size + input_size) * scale  # 망각 게이트 가중치
        self.b_f = np.ones(hidden_size)  # 망각 게이트 편향 (1로 초기화 = 초기에 잊지 않도록!)
        # ↑ b_f를 1로 초기화하는 이유: 학습 초기에 기울기가 잘 흐르도록
        #   Jozefowicz et al. (2015)의 권장 사항
        
        # Input Gate 가중치: W_i ∈ R^(H, H+d), b_i ∈ R^(H,)
        self.W_i = np.random.randn(hidden_size, hidden_size + input_size) * scale  # 입력 게이트 가중치
        self.b_i = np.zeros(hidden_size)  # 입력 게이트 편향 (0으로 초기화)
        
        # Cell Gate(후보 기억) 가중치: W_c ∈ R^(H, H+d), b_c ∈ R^(H,)
        self.W_c = np.random.randn(hidden_size, hidden_size + input_size) * scale  # 후보 기억 가중치
        self.b_c = np.zeros(hidden_size)  # 후보 기억 편향 (0으로 초기화)
        
        # Output Gate 가중치: W_o ∈ R^(H, H+d), b_o ∈ R^(H,)
        self.W_o = np.random.randn(hidden_size, hidden_size + input_size) * scale  # 출력 게이트 가중치
        self.b_o = np.zeros(hidden_size)  # 출력 게이트 편향 (0으로 초기화)
        
        # 기울기 저장용 딕셔너리 (backward에서 계산)
        self.grads = {}  # 가중치 기울기를 저장할 딕셔너리
        
        # 순전파 중간값 저장용 (역전파에서 사용)
        self.cache = {}  # 캐시: forward에서 저장, backward에서 사용
    
    def forward(self, x_t, h_prev, c_prev):
        """LSTM 셀의 순전파 (하나의 시점)
        
        입력:
            x_t    : 현재 시점의 입력 벡터, shape = (d,)
            h_prev : 이전 시점의 은닉 상태, shape = (H,)
            c_prev : 이전 시점의 셀 상태, shape = (H,)
            
        출력:
            h_t : 현재 시점의 은닉 상태, shape = (H,)
            c_t : 현재 시점의 셀 상태, shape = (H,)
        """
        H = self.hidden_size  # 은닉 차원 크기 (H)
        
        # --------------------------------------------------
        # Step 1: 입력 연결 (concatenation)
        # [h_{t-1}, x_t]를 하나의 벡터로 연결
        # --------------------------------------------------
        z = np.concatenate([h_prev, x_t])  # shape: (H + d,)
        # z = [h_{t-1} ; x_t] — 세미콜론은 벡터 연결을 의미
        
        # --------------------------------------------------
        # Step 2: Forget Gate (망각 게이트)
        # f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
        # "이전 기억 중 얼마나 잊을지" 결정
        # --------------------------------------------------
        f_t = sigmoid(np.dot(self.W_f, z) + self.b_f)  # shape: (H,), 값: 0~1
        # np.dot(W_f, z): 행렬-벡터 곱 (H, H+d) × (H+d,) = (H,)
        # + b_f: 편향 더하기
        # sigmoid: 0~1 사이로 변환 → "잊을 비율"
        
        # --------------------------------------------------
        # Step 3: Input Gate (입력 게이트)
        # i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
        # "새 정보를 얼마나 기억할지" 결정
        # --------------------------------------------------
        i_t = sigmoid(np.dot(self.W_i, z) + self.b_i)  # shape: (H,), 값: 0~1
        # 입력 게이트도 시그모이드 → 0~1 사이 = "기억할 비율"
        
        # --------------------------------------------------
        # Step 4: 후보 기억 (Candidate Cell State)
        # c_tilde = tanh(W_c · [h_{t-1}, x_t] + b_c)
        # "새로 만든 정보" (아직 필터링 전)
        # --------------------------------------------------
        c_tilde = np.tanh(np.dot(self.W_c, z) + self.b_c)  # shape: (H,), 값: -1~1
        # tanh: -1~1 사이 → "새 정보의 내용" (양수/음수 모두 가능)
        
        # --------------------------------------------------
        # Step 5: Cell State 업데이트 (★ LSTM의 핵심 수식 ★)
        # c_t = f_t ⊙ c_{t-1} + i_t ⊙ c_tilde
        # "기존 기억 일부 유지 + 새 기억 추가"
        # --------------------------------------------------
        c_t = f_t * c_prev + i_t * c_tilde  # shape: (H,)
        # f_t * c_prev: 기존 셀 상태에서 잊을 부분을 잊음 (원소별 곱)
        # i_t * c_tilde: 새 후보 기억에서 기억할 부분만 선택 (원소별 곱)
        # + : 덧셈! → 기울기가 직접 흐르는 "고속도로" 구조
        
        # --------------------------------------------------
        # Step 6: Output Gate (출력 게이트)
        # o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
        # "셀 상태에서 얼마나 출력할지" 결정
        # --------------------------------------------------
        o_t = sigmoid(np.dot(self.W_o, z) + self.b_o)  # shape: (H,), 값: 0~1
        # 출력 게이트도 시그모이드 → 0~1 사이 = "출력할 비율"
        
        # --------------------------------------------------
        # Step 7: Hidden State 계산 (최종 출력)
        # h_t = o_t ⊙ tanh(c_t)
        # "셀 상태를 정규화한 뒤, 출력 게이트로 필터링"
        # --------------------------------------------------
        tanh_c_t = np.tanh(c_t)  # 셀 상태를 -1~1 범위로 정규화
        h_t = o_t * tanh_c_t  # shape: (H,)
        # o_t * tanh(c_t): 출력 게이트가 결정한 비율만큼만 출력
        
        # --------------------------------------------------
        # 캐시 저장 (역전파에서 사용할 중간값들)
        # --------------------------------------------------
        self.cache = {
            'z': z,            # 연결된 입력 [h_{t-1}, x_t]
            'f_t': f_t,        # 망각 게이트 출력
            'i_t': i_t,        # 입력 게이트 출력
            'c_tilde': c_tilde,  # 후보 기억
            'c_t': c_t,        # 현재 셀 상태
            'o_t': o_t,        # 출력 게이트 출력
            'tanh_c_t': tanh_c_t,  # tanh(c_t)
            'h_prev': h_prev,  # 이전 은닉 상태
            'c_prev': c_prev,  # 이전 셀 상태
            'h_t': h_t,        # 현재 은닉 상태
        }
        
        return h_t, c_t  # 현재 시점의 은닉 상태와 셀 상태 반환
    
    def backward(self, dh_t, dc_t):
        """LSTM 셀의 역전파 (하나의 시점)
        
        입력:
            dh_t : h_t에 대한 손실의 기울기, shape = (H,)
            dc_t : c_t에 대한 손실의 기울기 (다음 시점에서 전달됨), shape = (H,)
            
        출력:
            dx_t   : x_t에 대한 기울기, shape = (d,)
            dh_prev: h_{t-1}에 대한 기울기, shape = (H,)
            dc_prev: c_{t-1}에 대한 기울기, shape = (H,)
        """
        # 캐시에서 순전파 중간값 불러오기
        z = self.cache['z']              # 연결된 입력
        f_t = self.cache['f_t']          # 망각 게이트 출력
        i_t = self.cache['i_t']          # 입력 게이트 출력
        c_tilde = self.cache['c_tilde']  # 후보 기억
        c_t = self.cache['c_t']          # 현재 셀 상태
        o_t = self.cache['o_t']          # 출력 게이트 출력
        tanh_c_t = self.cache['tanh_c_t']  # tanh(c_t)
        h_prev = self.cache['h_prev']    # 이전 은닉 상태
        c_prev = self.cache['c_prev']    # 이전 셀 상태
        
        H = self.hidden_size  # 은닉 차원 크기
        
        # --------------------------------------------------
        # 역전파 Step 1: Output Gate와 tanh(c_t)의 기울기
        # h_t = o_t ⊙ tanh(c_t) 에서:
        # --------------------------------------------------
        # dh_t/do_t = tanh(c_t)  (h_t = o_t * tanh_c_t에서 o_t에 대한 미분)
        # dh_t/d(tanh_c_t) = o_t
        
        # Output Gate의 시그모이드 이전 값에 대한 기울기
        do_t = dh_t * tanh_c_t  # h_t에서 o_t 방향으로의 기울기
        do_raw = do_t * sigmoid_derivative(o_t)  # 시그모이드 미분 적용
        # sigmoid_derivative(o_t) = o_t * (1 - o_t)
        
        # Cell State에 대한 기울기 (두 경로에서 합산)
        # 경로 1: h_t = o_t * tanh(c_t) 에서 c_t로의 기울기
        dc_from_h = dh_t * o_t * tanh_derivative(tanh_c_t)  # tanh 미분 적용
        # 경로 2: 다음 시점에서 직접 전달된 c_t에 대한 기울기
        dc_total = dc_from_h + dc_t  # 두 경로의 기울기를 합산
        # ↑ 이것이 "기울기 고속도로"의 핵심!
        
        # --------------------------------------------------
        # 역전파 Step 2: Cell State 업데이트 수식의 기울기
        # c_t = f_t ⊙ c_{t-1} + i_t ⊙ c_tilde 에서:
        # --------------------------------------------------
        
        # Forget Gate에 대한 기울기
        # dc_t/df_t = c_{t-1}
        df_t = dc_total * c_prev  # 망각 게이트 방향 기울기
        df_raw = df_t * sigmoid_derivative(f_t)  # 시그모이드 미분 적용
        
        # Input Gate에 대한 기울기
        # dc_t/di_t = c_tilde
        di_t = dc_total * c_tilde  # 입력 게이트 방향 기울기
        di_raw = di_t * sigmoid_derivative(i_t)  # 시그모이드 미분 적용
        
        # 후보 기억(c_tilde)에 대한 기울기
        # dc_t/dc_tilde = i_t
        dc_tilde = dc_total * i_t  # 후보 기억 방향 기울기
        dc_raw = dc_tilde * tanh_derivative(c_tilde)  # tanh 미분 적용
        
        # 이전 셀 상태(c_{t-1})에 대한 기울기 ★
        # dc_t/dc_{t-1} = f_t  ← 이것이 기울기 고속도로!
        dc_prev = dc_total * f_t  # f_t가 1에 가까우면 기울기가 거의 그대로 전달!
        
        # --------------------------------------------------
        # 역전파 Step 3: 가중치에 대한 기울기
        # 각 게이트: raw = W · z + b 에서 W와 b에 대한 미분
        # --------------------------------------------------
        
        # Forget Gate 가중치 기울기
        self.grads['dW_f'] = np.outer(df_raw, z)  # 외적: (H,) × (H+d,) = (H, H+d)
        self.grads['db_f'] = df_raw  # 편향 기울기는 raw 기울기와 동일
        
        # Input Gate 가중치 기울기
        self.grads['dW_i'] = np.outer(di_raw, z)  # 외적으로 가중치 기울기 계산
        self.grads['db_i'] = di_raw  # 편향 기울기
        
        # Cell Gate(후보 기억) 가중치 기울기
        self.grads['dW_c'] = np.outer(dc_raw, z)  # 외적으로 가중치 기울기 계산
        self.grads['db_c'] = dc_raw  # 편향 기울기
        
        # Output Gate 가중치 기울기
        self.grads['dW_o'] = np.outer(do_raw, z)  # 외적으로 가중치 기울기 계산
        self.grads['db_o'] = do_raw  # 편향 기울기
        
        # --------------------------------------------------
        # 역전파 Step 4: 입력(z)에 대한 기울기
        # z = [h_{t-1}, x_t] 이므로, dz를 분리하면 dh_prev와 dx_t를 얻음
        # --------------------------------------------------
        # 4개 게이트 모두에서 z로 흘러오는 기울기를 합산
        dz = (np.dot(self.W_f.T, df_raw) +  # Forget Gate에서 z로
              np.dot(self.W_i.T, di_raw) +   # Input Gate에서 z로
              np.dot(self.W_c.T, dc_raw) +   # Cell Gate에서 z로
              np.dot(self.W_o.T, do_raw))    # Output Gate에서 z로
        
        # z = [h_prev, x_t]를 다시 분리
        dh_prev = dz[:H]   # 처음 H개 원소 = h_{t-1}에 대한 기울기
        dx_t = dz[H:]      # 나머지 d개 원소 = x_t에 대한 기울기
        
        return dx_t, dh_prev, dc_prev  # 입력, 이전 은닉, 이전 셀에 대한 기울기

# LSTMCell 생성 테스트
input_size = 3   # 입력 차원 d = 3 (예: 3차원 입력 벡터)
hidden_size = 4  # 은닉 차원 H = 4

lstm_cell = LSTMCell(input_size, hidden_size)  # LSTMCell 인스턴스 생성

print("=== LSTMCell 생성 완료 ===")
print(f"입력 차원 (d): {input_size}")  # 입력 차원 출력
print(f"은닉 차원 (H): {hidden_size}")  # 은닉 차원 출력
print(f"W_f 크기: {lstm_cell.W_f.shape} = (H, H+d) = ({hidden_size}, {hidden_size+input_size})")  # 망각 게이트 가중치 크기
print(f"b_f 크기: {lstm_cell.b_f.shape} = (H,) = ({hidden_size},)")  # 망각 게이트 편향 크기
print(f"b_f 초기값: {lstm_cell.b_f} (1로 초기화 → 초기에 잊지 않도록!)")  # 편향 초기값 확인
total_params = 4 * hidden_size * (hidden_size + input_size) + 4 * hidden_size  # 총 파라미터 수 계산
print(f"\n총 파라미터 수: {total_params}")
print(f"  = 4개 게이트 × (가중치 H×(H+d) + 편향 H)")
print(f"  = 4 × ({hidden_size}×{hidden_size+input_size} + {hidden_size})")
print(f"  = 4 × ({hidden_size*(hidden_size+input_size)} + {hidden_size}) = {total_params}")

In [ ]:
# ============================================================
# Part 2.3: Forward 예제 - 각 게이트 값을 직접 확인합니다
# 작은 예제로 LSTM 셀의 동작을 한 단계씩 추적합니다
# ============================================================

# 테스트용 입력 데이터 생성
np.random.seed(42)  # 재현성을 위한 시드 고정

# 입력 벡터: 3차원 (예: 단어 임베딩이 3차원이라고 가정)
x_t = np.array([0.5, -0.3, 0.8])  # 현재 시점의 입력
print(f"입력 x_t: {x_t}, shape: {x_t.shape}")  # 입력 확인

# 이전 시점의 상태 (처음에는 0으로 초기화)
h_prev = np.zeros(hidden_size)  # 이전 은닉 상태 (모두 0)
c_prev = np.zeros(hidden_size)  # 이전 셀 상태 (모두 0)
print(f"이전 은닉 상태 h_{{t-1}}: {h_prev}")  # 이전 은닉 상태 확인
print(f"이전 셀 상태 c_{{t-1}}: {c_prev}")  # 이전 셀 상태 확인

# LSTM 셀 순전파 실행
h_t, c_t = lstm_cell.forward(x_t, h_prev, c_prev)  # forward 호출

# 각 게이트의 출력값을 캐시에서 꺼내서 확인
cache = lstm_cell.cache  # 캐시 접근

print("\n" + "=" * 60)
print("🔍 각 게이트의 출력값 상세 분석")
print("=" * 60)

print(f"\n[Step 1] 연결 벡터 z = [h_{{t-1}}, x_t]:")
print(f"  z = {cache['z'].round(4)}, shape: {cache['z'].shape}")  # 연결 벡터 확인

print(f"\n[Step 2] Forget Gate (망각 게이트):")
print(f"  f_t = σ(W_f · z + b_f) = {cache['f_t'].round(4)}")  # 망각 게이트 출력
print(f"  해석: 각 원소가 0~1 → 이전 기억을 {cache['f_t'].round(2)} 비율로 유지")

print(f"\n[Step 3] Input Gate (입력 게이트):")
print(f"  i_t = σ(W_i · z + b_i) = {cache['i_t'].round(4)}")  # 입력 게이트 출력
print(f"  해석: 새 정보를 {cache['i_t'].round(2)} 비율로 반영")

print(f"\n[Step 4] 후보 기억 (Candidate Cell State):")
print(f"  c̃_t = tanh(W_c · z + b_c) = {cache['c_tilde'].round(4)}")  # 후보 기억 출력
print(f"  해석: 새로 만든 정보 (양수/음수 모두 가능)")

print(f"\n[Step 5] Cell State 업데이트 (★핵심★):")
print(f"  c_t = f_t ⊙ c_{{t-1}} + i_t ⊙ c̃_t")
print(f"      = {cache['f_t'].round(4)} ⊙ {c_prev.round(4)} + {cache['i_t'].round(4)} ⊙ {cache['c_tilde'].round(4)}")
print(f"      = {(cache['f_t'] * c_prev).round(4)} + {(cache['i_t'] * cache['c_tilde']).round(4)}")
print(f"      = {c_t.round(4)}")

print(f"\n[Step 6] Output Gate (출력 게이트):")
print(f"  o_t = σ(W_o · z + b_o) = {cache['o_t'].round(4)}")  # 출력 게이트 출력

print(f"\n[Step 7] Hidden State (최종 출력):")
print(f"  h_t = o_t ⊙ tanh(c_t)")
print(f"      = {cache['o_t'].round(4)} ⊙ tanh({c_t.round(4)})")
print(f"      = {cache['o_t'].round(4)} ⊙ {cache['tanh_c_t'].round(4)}")
print(f"      = {h_t.round(4)}")

In [ ]:
# ============================================================
# Part 2.4: 시퀀스 처리 및 게이트 값 시각화
# 여러 시점의 입력을 LSTM 셀에 순차적으로 넣고,
# 각 시점에서 게이트 값의 변화를 시각화합니다.
# ============================================================

# 시퀀스 데이터 생성 (10개 시점, 각 3차원 입력)
np.random.seed(123)  # 재현성을 위한 시드 고정
seq_len = 10  # 시퀀스 길이 (10개 시점)
input_dim = 3  # 입력 차원

# 간단한 패턴이 있는 시퀀스 생성
# 사인파 + 노이즈로 구성 (패턴 학습을 위해)
t_steps = np.linspace(0, 2 * np.pi, seq_len)  # 0~2π를 10등분
sequence = np.column_stack([  # 3개 채널의 시퀀스 생성
    np.sin(t_steps),          # 채널 1: 사인파
    np.cos(t_steps),          # 채널 2: 코사인파
    np.sin(2 * t_steps) * 0.5  # 채널 3: 2배 주파수 사인파 (절반 진폭)
])
print(f"시퀀스 shape: {sequence.shape} = (시퀀스길이, 입력차원)")  # 시퀀스 형태 확인

# 새로운 LSTMCell 생성
lstm_cell = LSTMCell(input_dim, hidden_size)  # 입력 3차원, 은닉 4차원

# 각 시점의 게이트 값을 저장할 리스트
forget_gates = []   # 망각 게이트 값 저장용
input_gates = []    # 입력 게이트 값 저장용
output_gates = []   # 출력 게이트 값 저장용
cell_states = []    # 셀 상태 값 저장용
hidden_states = []  # 은닉 상태 값 저장용

# 초기 상태 설정
h = np.zeros(hidden_size)  # 초기 은닉 상태 (모두 0)
c = np.zeros(hidden_size)  # 초기 셀 상태 (모두 0)

# 시퀀스의 각 시점을 순차적으로 처리
print("\n=== 시퀀스 순전파 진행 ===")
for t in range(seq_len):  # 각 시점에 대해
    x = sequence[t]  # 현재 시점의 입력 벡터 (3차원)
    h, c = lstm_cell.forward(x, h, c)  # LSTM 셀 순전파 실행
    
    # 각 게이트 값을 리스트에 저장
    forget_gates.append(lstm_cell.cache['f_t'].copy())   # 망각 게이트 값 저장
    input_gates.append(lstm_cell.cache['i_t'].copy())    # 입력 게이트 값 저장
    output_gates.append(lstm_cell.cache['o_t'].copy())   # 출력 게이트 값 저장
    cell_states.append(c.copy())    # 셀 상태 저장
    hidden_states.append(h.copy())  # 은닉 상태 저장
    
    print(f"  시점 {t}: f_t 평균={lstm_cell.cache['f_t'].mean():.4f}, "  # 각 시점 정보 출력
          f"i_t 평균={lstm_cell.cache['i_t'].mean():.4f}, "
          f"o_t 평균={lstm_cell.cache['o_t'].mean():.4f}")

# numpy 배열로 변환 (시각화를 위해)
forget_gates = np.array(forget_gates)   # shape: (seq_len, hidden_size)
input_gates = np.array(input_gates)     # shape: (seq_len, hidden_size)
output_gates = np.array(output_gates)   # shape: (seq_len, hidden_size)
cell_states = np.array(cell_states)     # shape: (seq_len, hidden_size)
hidden_states = np.array(hidden_states)  # shape: (seq_len, hidden_size)

# ============================================================
# 게이트 값 히트맵 시각화
# 각 시점(x축) × 각 은닉 유닛(y축)의 게이트 값을 색으로 표현
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2행 3열 서브플롯

# 1. Forget Gate 히트맵
im1 = axes[0, 0].imshow(forget_gates.T, aspect='auto', cmap='RdYlGn',  # 빨-노-초 컬러맵
                         vmin=0, vmax=1)  # 0~1 범위로 고정
axes[0, 0].set_title('Forget Gate (f_t)\n0=잊음(빨강), 1=기억(초록)', fontsize=12)  # 제목
axes[0, 0].set_xlabel('시점 (t)')  # x축
axes[0, 0].set_ylabel('은닉 유닛 번호')  # y축
plt.colorbar(im1, ax=axes[0, 0])  # 컬러바 추가

# 2. Input Gate 히트맵
im2 = axes[0, 1].imshow(input_gates.T, aspect='auto', cmap='RdYlGn',  # 빨-노-초 컬러맵
                         vmin=0, vmax=1)
axes[0, 1].set_title('Input Gate (i_t)\n0=무시(빨강), 1=반영(초록)', fontsize=12)  # 제목
axes[0, 1].set_xlabel('시점 (t)')  # x축
axes[0, 1].set_ylabel('은닉 유닛 번호')  # y축
plt.colorbar(im2, ax=axes[0, 1])  # 컬러바 추가

# 3. Output Gate 히트맵
im3 = axes[0, 2].imshow(output_gates.T, aspect='auto', cmap='RdYlGn',  # 빨-노-초 컬러맵
                         vmin=0, vmax=1)
axes[0, 2].set_title('Output Gate (o_t)\n0=출력안함(빨강), 1=출력(초록)', fontsize=12)  # 제목
axes[0, 2].set_xlabel('시점 (t)')  # x축
axes[0, 2].set_ylabel('은닉 유닛 번호')  # y축
plt.colorbar(im3, ax=axes[0, 2])  # 컬러바 추가

# 4. Cell State 변화
for unit in range(hidden_size):  # 각 은닉 유닛별로
    axes[1, 0].plot(range(seq_len), cell_states[:, unit],  # 셀 상태 그래프
                    label=f'유닛 {unit}', linewidth=2)
axes[1, 0].set_title('Cell State (c_t) 변화', fontsize=12)  # 제목
axes[1, 0].set_xlabel('시점 (t)')  # x축
axes[1, 0].set_ylabel('셀 상태 값')  # y축
axes[1, 0].legend(fontsize=9)  # 범례
axes[1, 0].grid(True, alpha=0.3)  # 격자

# 5. Hidden State 변화
for unit in range(hidden_size):  # 각 은닉 유닛별로
    axes[1, 1].plot(range(seq_len), hidden_states[:, unit],  # 은닉 상태 그래프
                    label=f'유닛 {unit}', linewidth=2)
axes[1, 1].set_title('Hidden State (h_t) 변화', fontsize=12)  # 제목
axes[1, 1].set_xlabel('시점 (t)')  # x축
axes[1, 1].set_ylabel('은닉 상태 값')  # y축
axes[1, 1].legend(fontsize=9)  # 범례
axes[1, 1].grid(True, alpha=0.3)  # 격자

# 6. 게이트 평균값 비교
axes[1, 2].plot(range(seq_len), forget_gates.mean(axis=1), 'r-o', label='Forget Gate', linewidth=2)  # 망각 게이트 평균
axes[1, 2].plot(range(seq_len), input_gates.mean(axis=1), 'g-s', label='Input Gate', linewidth=2)  # 입력 게이트 평균
axes[1, 2].plot(range(seq_len), output_gates.mean(axis=1), 'b-^', label='Output Gate', linewidth=2)  # 출력 게이트 평균
axes[1, 2].set_title('게이트 평균값 비교', fontsize=12)  # 제목
axes[1, 2].set_xlabel('시점 (t)')  # x축
axes[1, 2].set_ylabel('게이트 값 (0~1)')  # y축
axes[1, 2].legend(fontsize=10)  # 범례
axes[1, 2].grid(True, alpha=0.3)  # 격자
axes[1, 2].set_ylim(-0.05, 1.05)  # y축 범위

plt.suptitle('LSTM 게이트 값 시각화 (시퀀스 처리 중)', fontsize=16, y=1.02)  # 전체 제목
plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print("\n🔑 관찰 포인트:")
print("  1. Forget Gate: 어떤 유닛이 정보를 유지하고, 어떤 유닛이 잊는지 확인")
print("  2. Input Gate: 어느 시점에서 새 정보를 많이 받아들이는지 확인")
print("  3. Cell State: 장기 기억이 어떻게 누적되는지 확인")
print("  4. Hidden State: Cell State와 비교하여 출력이 어떻게 다른지 확인")

In [ ]:
# ============================================================
# Part 2.5: 역전파 검증 - 수치 기울기(Numerical Gradient)와 비교
# 우리가 구현한 backward가 올바른지 확인합니다.
# 수치 미분과 해석적 미분이 일치하면 구현이 정확한 것입니다.
# ============================================================

def numerical_gradient_check(lstm_cell, x_t, h_prev, c_prev, epsilon=1e-5):
    """수치 기울기 검증 함수
    
    중심차분법(central difference)으로 기울기를 근사하고,
    backward()의 해석적 기울기와 비교합니다.
    
    Args:
        lstm_cell: LSTMCell 인스턴스
        x_t: 입력 벡터
        h_prev: 이전 은닉 상태
        c_prev: 이전 셀 상태
        epsilon: 수치 미분에 사용할 작은 값
    """
    # 먼저 순전파 실행
    h_t, c_t = lstm_cell.forward(x_t, h_prev, c_prev)  # 순전파 실행
    
    # 손실 = h_t의 모든 원소 합 (간단한 테스트용 손실 함수)
    loss = np.sum(h_t)  # 손실 계산 (단순 합)
    
    # 해석적 기울기 계산 (backward)
    dh_t = np.ones_like(h_t)  # dL/dh_t = 1 (sum의 미분)
    dc_t = np.zeros_like(c_t)  # 마지막 시점이므로 dc_t = 0
    dx_t_anal, dh_prev_anal, dc_prev_anal = lstm_cell.backward(dh_t, dc_t)  # 역전파 실행
    
    print("=== 수치 기울기 검증 ===")
    print(f"손실 L = sum(h_t) = {loss:.6f}")  # 손실값 출력
    
    # W_f에 대한 수치 기울기 검증 (일부 원소만)
    param_name = 'W_f'  # 검증할 파라미터
    param = lstm_cell.W_f  # 파라미터 참조
    grad_analytical = lstm_cell.grads['dW_f']  # 해석적 기울기
    
    print(f"\n{param_name} 기울기 검증 (처음 3개 원소):")
    
    num_checks = 3  # 검증할 원소 수
    checked = 0  # 검증 완료 수
    max_error = 0  # 최대 오차
    
    for idx in np.ndindex(param.shape):  # 모든 인덱스에 대해
        if checked >= num_checks:  # 지정된 수만큼만 검증
            break
        
        # 중심차분법: f'(x) ≈ (f(x+ε) - f(x-ε)) / (2ε)
        original = param[idx]  # 원래 값 저장
        
        # f(x + ε)
        param[idx] = original + epsilon  # ε만큼 증가
        h_plus, _ = lstm_cell.forward(x_t, h_prev, c_prev)  # 순전파 (증가)
        loss_plus = np.sum(h_plus)  # 손실 계산 (증가)
        
        # f(x - ε)
        param[idx] = original - epsilon  # ε만큼 감소
        h_minus, _ = lstm_cell.forward(x_t, h_prev, c_prev)  # 순전파 (감소)
        loss_minus = np.sum(h_minus)  # 손실 계산 (감소)
        
        # 원래 값 복원
        param[idx] = original  # 파라미터 복원
        
        # 수치 기울기 계산
        grad_numerical = (loss_plus - loss_minus) / (2 * epsilon)  # 중심차분법
        
        # 해석적 기울기와 비교
        grad_anal = grad_analytical[idx]  # 해석적 기울기
        
        # 상대 오차 계산
        error = abs(grad_numerical - grad_anal) / (abs(grad_numerical) + abs(grad_anal) + 1e-8)  # 상대 오차
        max_error = max(max_error, error)  # 최대 오차 갱신
        
        status = "✓" if error < 1e-5 else "✗"  # 통과/실패 표시
        print(f"  {param_name}{list(idx)}: 수치={grad_numerical:.8f}, "  # 결과 출력
              f"해석={grad_anal:.8f}, 오차={error:.2e} {status}")
        
        checked += 1  # 검증 완료 수 증가
    
    # 순전파 다시 실행 (캐시 복원)
    lstm_cell.forward(x_t, h_prev, c_prev)  # 순전파 재실행
    lstm_cell.backward(dh_t, dc_t)  # 역전파 재실행
    
    print(f"\n최대 상대 오차: {max_error:.2e}")
    if max_error < 1e-5:  # 오차가 충분히 작은지 확인
        print("✓ 역전파 구현이 정확합니다! (오차 < 1e-5)")
    else:
        print("✗ 역전파에 문제가 있을 수 있습니다.")

# 수치 기울기 검증 실행
np.random.seed(42)  # 재현성을 위한 시드 고정
lstm_test = LSTMCell(input_size=3, hidden_size=4)  # 테스트용 LSTM 셀 생성
x_test = np.random.randn(3)      # 랜덤 입력 생성
h_test = np.random.randn(4) * 0.1  # 작은 랜덤 은닉 상태
c_test = np.random.randn(4) * 0.1  # 작은 랜덤 셀 상태

numerical_gradient_check(lstm_test, x_test, h_test, c_test)  # 검증 실행

---

## Part 2.6: LSTM으로 간단한 시퀀스 학습하기

이제 구현한 LSTMCell을 사용하여 **간단한 패턴을 학습**해봅니다.

**과제**: 입력 시퀀스의 마지막 값을 예측하기 (Many-to-One)
- 예: [0.1, 0.3, 0.5, 0.7] → 0.9 예측 (등차수열 패턴)

In [ ]:
# ============================================================
# Part 2.6: LSTM으로 간단한 시퀀스 패턴 학습
# 등차수열의 다음 값을 예측하는 문제를 풀어봅니다
# ============================================================

class SimpleLSTM:
    """간단한 LSTM 네트워크 (Many-to-One)
    
    여러 시점의 입력을 받아 하나의 값을 예측합니다.
    LSTMCell + 출력 레이어(Linear)로 구성됩니다.
    
    Args:
        input_size: 입력 차원
        hidden_size: 은닉 차원
        output_size: 출력 차원
    """
    
    def __init__(self, input_size, hidden_size, output_size):
        """네트워크 초기화"""
        self.lstm_cell = LSTMCell(input_size, hidden_size)  # LSTM 셀 생성
        self.hidden_size = hidden_size  # 은닉 차원 저장
        
        # 출력 레이어: h_t → 예측값
        scale = np.sqrt(2.0 / (hidden_size + output_size))  # Xavier 초기화 스케일
        self.W_out = np.random.randn(output_size, hidden_size) * scale  # 출력 가중치 (1, H)
        self.b_out = np.zeros(output_size)  # 출력 편향
    
    def forward(self, sequence):
        """시퀀스 전체를 순전파합니다
        
        Args:
            sequence: 입력 시퀀스, shape = (seq_len, input_size)
        Returns:
            prediction: 예측값, shape = (output_size,)
        """
        seq_len = len(sequence)  # 시퀀스 길이
        
        # 초기 상태
        h = np.zeros(self.hidden_size)  # 초기 은닉 상태
        c = np.zeros(self.hidden_size)  # 초기 셀 상태
        
        # 캐시 저장용 (역전파에서 사용)
        self.caches = []  # 각 시점의 캐시 저장
        self.h_states = [h.copy()]  # 은닉 상태 히스토리 (h_0 포함)
        self.c_states = [c.copy()]  # 셀 상태 히스토리 (c_0 포함)
        self.inputs = sequence.copy()  # 입력 시퀀스 저장
        
        # 각 시점별 순전파
        for t in range(seq_len):  # 각 시점에 대해
            h, c = self.lstm_cell.forward(sequence[t], h, c)  # LSTM 셀 순전파
            self.caches.append(dict(self.lstm_cell.cache))  # 캐시 복사하여 저장
            self.h_states.append(h.copy())  # 은닉 상태 저장
            self.c_states.append(c.copy())  # 셀 상태 저장
        
        # 마지막 시점의 은닉 상태로 예측
        self.final_h = h  # 마지막 은닉 상태 저장
        prediction = np.dot(self.W_out, h) + self.b_out  # 선형 변환: (1, H) × (H,) + (1,) = (1,)
        
        return prediction  # 예측값 반환
    
    def backward(self, d_prediction):
        """전체 시퀀스에 대한 역전파
        
        Args:
            d_prediction: 예측값에 대한 손실의 기울기
        """
        seq_len = len(self.inputs)  # 시퀀스 길이
        
        # 출력 레이어의 기울기
        self.dW_out = np.outer(d_prediction, self.final_h)  # 출력 가중치 기울기
        self.db_out = d_prediction.copy()  # 출력 편향 기울기
        
        # LSTM으로 전달되는 기울기 초기화
        dh = np.dot(self.W_out.T, d_prediction)  # h에 대한 기울기: (H, 1) × (1,) = (H,)
        dc = np.zeros(self.hidden_size)  # 마지막 시점이므로 dc = 0
        
        # 시간 역순으로 역전파 (BPTT: Backpropagation Through Time)
        # 기울기 누적용 딕셔너리 초기화
        total_grads = {  # 모든 시점의 기울기를 합산할 딕셔너리
            'dW_f': np.zeros_like(self.lstm_cell.W_f),
            'db_f': np.zeros_like(self.lstm_cell.b_f),
            'dW_i': np.zeros_like(self.lstm_cell.W_i),
            'db_i': np.zeros_like(self.lstm_cell.b_i),
            'dW_c': np.zeros_like(self.lstm_cell.W_c),
            'db_c': np.zeros_like(self.lstm_cell.b_c),
            'dW_o': np.zeros_like(self.lstm_cell.W_o),
            'db_o': np.zeros_like(self.lstm_cell.b_o),
        }
        
        for t in reversed(range(seq_len)):  # 시간 역순 (t-1, t-2, ..., 0)
            # 해당 시점의 캐시를 LSTM 셀에 복원
            self.lstm_cell.cache = self.caches[t]  # 캐시 복원
            
            # 해당 시점의 역전파 실행
            dx, dh, dc = self.lstm_cell.backward(dh, dc)  # 역전파 실행
            # dh: 이전 시점 은닉 상태에 대한 기울기
            # dc: 이전 시점 셀 상태에 대한 기울기 (★ 기울기 고속도로!)
            
            # 기울기 누적 (모든 시점의 기울기를 합산)
            for key in total_grads:  # 각 파라미터에 대해
                total_grads[key] += self.lstm_cell.grads[key.replace('d', '', 1) if key.startswith('d') else key]
            # 수정: 올바른 키 매핑
            total_grads['dW_f'] = total_grads['dW_f'] - self.lstm_cell.grads.get('dW_f', np.zeros_like(self.lstm_cell.W_f)) + self.lstm_cell.grads.get('dW_f', np.zeros_like(self.lstm_cell.W_f))
        
        # LSTM 셀 기울기를 누적값으로 설정
        self.lstm_cell.grads = total_grads  # 전체 기울기 설정
    
    def update_params(self, lr):
        """SGD로 파라미터 업데이트
        
        Args:
            lr: 학습률 (learning rate)
        """
        # LSTM 셀 파라미터 업데이트
        self.lstm_cell.W_f -= lr * self.lstm_cell.grads.get('dW_f', 0)  # 망각 게이트 가중치 갱신
        self.lstm_cell.b_f -= lr * self.lstm_cell.grads.get('db_f', 0)  # 망각 게이트 편향 갱신
        self.lstm_cell.W_i -= lr * self.lstm_cell.grads.get('dW_i', 0)  # 입력 게이트 가중치 갱신
        self.lstm_cell.b_i -= lr * self.lstm_cell.grads.get('db_i', 0)  # 입력 게이트 편향 갱신
        self.lstm_cell.W_c -= lr * self.lstm_cell.grads.get('dW_c', 0)  # 후보 기억 가중치 갱신
        self.lstm_cell.b_c -= lr * self.lstm_cell.grads.get('db_c', 0)  # 후보 기억 편향 갱신
        self.lstm_cell.W_o -= lr * self.lstm_cell.grads.get('dW_o', 0)  # 출력 게이트 가중치 갱신
        self.lstm_cell.b_o -= lr * self.lstm_cell.grads.get('db_o', 0)  # 출력 게이트 편향 갱신
        
        # 출력 레이어 파라미터 업데이트
        self.W_out -= lr * self.dW_out  # 출력 가중치 갱신
        self.b_out -= lr * self.db_out  # 출력 편향 갱신


# ============================================================
# 학습 데이터 생성: 등차수열 패턴
# ============================================================
print("=== 등차수열 패턴 학습 ===\n")

# 데이터 생성 함수
def generate_arithmetic_sequence(n_samples, seq_len, noise=0.02):
    """등차수열 데이터 생성
    
    Args:
        n_samples: 샘플 수
        seq_len: 시퀀스 길이 (입력 길이, 마지막 1개는 타겟)
        noise: 노이즈 크기
    Returns:
        X: 입력 시퀀스, shape = (n_samples, seq_len, 1)
        y: 타겟 값, shape = (n_samples, 1)
    """
    X = []  # 입력 리스트
    y = []  # 타겟 리스트
    for _ in range(n_samples):  # 샘플 수만큼 반복
        start = np.random.uniform(-1, 1)  # 시작값: -1~1 사이 랜덤
        step = np.random.uniform(0.05, 0.3)  # 공차: 0.05~0.3 사이 랜덤
        seq = start + step * np.arange(seq_len + 1)  # 등차수열 생성 (입력 + 타겟)
        seq += np.random.randn(seq_len + 1) * noise  # 약간의 노이즈 추가
        X.append(seq[:seq_len].reshape(-1, 1))  # 입력: 처음~마지막-1
        y.append(seq[seq_len:seq_len+1])  # 타겟: 마지막 값
    return np.array(X), np.array(y)  # numpy 배열로 반환

# 데이터 생성
np.random.seed(42)  # 재현성
n_train = 200  # 학습 데이터 수
seq_length = 5  # 시퀀스 길이
X_train, y_train = generate_arithmetic_sequence(n_train, seq_length)  # 학습 데이터 생성

print(f"학습 데이터 X shape: {X_train.shape} = (샘플수, 시퀀스길이, 입력차원)")  # 데이터 형태
print(f"타겟 y shape: {y_train.shape} = (샘플수, 출력차원)")  # 타겟 형태
print(f"\n예시 입력: {X_train[0].flatten().round(3)}")  # 첫 번째 샘플 입력
print(f"예시 타겟: {y_train[0].round(3)}")  # 첫 번째 샘플 타겟

In [ ]:
# ============================================================
# Part 2.6 (계속): LSTM 학습 루프
# 위에서 생성한 데이터로 실제 학습을 진행합니다
# ============================================================

# 모델 생성
np.random.seed(42)  # 재현성
model = SimpleLSTM(input_size=1, hidden_size=8, output_size=1)  # LSTM 모델 생성
# input_size=1: 입력 차원 (스칼라 값)
# hidden_size=8: 은닉 차원 (8개 유닛)
# output_size=1: 출력 차원 (다음 값 하나 예측)

# 학습 하이퍼파라미터
learning_rate = 0.001  # 학습률
n_epochs = 50  # 에폭 수
losses = []  # 에폭별 손실 저장

print("=== LSTM 학습 시작 ===\n")

for epoch in range(n_epochs):  # 각 에폭에 대해
    epoch_loss = 0  # 에폭 손실 초기화
    
    for i in range(n_train):  # 각 학습 샘플에 대해
        # 순전파
        pred = model.forward(X_train[i])  # 예측값 계산
        
        # MSE 손실 계산: L = (pred - target)²
        target = y_train[i]  # 타겟 값
        loss = np.sum((pred - target) ** 2)  # MSE 손실
        epoch_loss += loss  # 에폭 손실 누적
        
        # 역전파
        d_pred = 2 * (pred - target)  # MSE의 미분: dL/dpred = 2(pred - target)
        model.backward(d_pred)  # 역전파 실행
        
        # 파라미터 업데이트
        model.update_params(learning_rate)  # SGD 업데이트
    
    avg_loss = epoch_loss / n_train  # 평균 손실
    losses.append(avg_loss)  # 손실 저장
    
    if (epoch + 1) % 10 == 0:  # 10 에폭마다 출력
        print(f"  에폭 {epoch+1:3d}/{n_epochs}: 평균 MSE = {avg_loss:.6f}")

# 학습 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1행 2열 서브플롯

# 손실 그래프
axes[0].plot(range(1, n_epochs+1), losses, 'b-', linewidth=2)  # 손실 변화
axes[0].set_xlabel('에폭', fontsize=12)  # x축
axes[0].set_ylabel('평균 MSE 손실', fontsize=12)  # y축
axes[0].set_title('LSTM 학습 곡선 (밑바닥 구현)', fontsize=14)  # 제목
axes[0].grid(True, alpha=0.3)  # 격자

# 예측 vs 실제 비교
n_test = 20  # 테스트 샘플 수
predictions = []  # 예측값 저장
targets = []  # 타겟 저장

for i in range(n_test):  # 테스트 샘플에 대해
    pred = model.forward(X_train[i])  # 예측
    predictions.append(pred[0])  # 예측값 저장
    targets.append(y_train[i][0])  # 타겟 저장

axes[1].scatter(targets, predictions, alpha=0.7, s=50, label='예측')  # 산점도
axes[1].plot([min(targets), max(targets)], [min(targets), max(targets)],  # 대각선 (완벽한 예측 기준)
            'r--', linewidth=2, label='완벽한 예측')
axes[1].set_xlabel('실제 값', fontsize=12)  # x축
axes[1].set_ylabel('예측 값', fontsize=12)  # y축
axes[1].set_title('예측 vs 실제 비교', fontsize=14)  # 제목
axes[1].legend(fontsize=11)  # 범례
axes[1].grid(True, alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print("\n✅ 밑바닥 구현 LSTM으로 등차수열 다음 값 예측 학습 완료!")
print("대각선에 가까울수록 예측이 정확합니다.")

---

# Part 3: PyTorch nn.LSTM 사용

> Part 2에서 밑바닥부터 구현한 LSTM을 이제 PyTorch의 `nn.LSTM`으로 훨씬 간단하게 사용합니다.
> 또한 **감정 분류(Sentiment Classification)** 예제를 통해 실전 활용법을 배웁니다.

---

## 3.1 nn.LSTM 파라미터 완전 해부

In [ ]:
# ============================================================
# Part 3.1: PyTorch nn.LSTM 파라미터 상세 설명
# nn.LSTM의 각 파라미터가 무엇을 의미하는지 하나씩 확인합니다
# ============================================================

import torch  # PyTorch 불러오기
import torch.nn as nn  # 신경망 모듈

# nn.LSTM 생성 (모든 파라미터를 명시적으로 지정)
lstm_layer = nn.LSTM(
    input_size=10,      # 입력 벡터의 차원 (d) — 예: 단어 임베딩 크기
    hidden_size=20,     # 은닉 상태의 차원 (H) — LSTM의 "기억 용량"
    num_layers=1,       # LSTM 층의 수 — 깊은 LSTM을 만들 때 2 이상 사용
    bias=True,          # 편향(bias) 사용 여부 — 보통 True
    batch_first=True,   # 입력 텐서의 형태: True면 (배치, 시퀀스길이, 입력차원)
    dropout=0.0,        # 드롭아웃 비율 — num_layers > 1일 때만 의미 있음
    bidirectional=False  # 양방향 LSTM 여부 — True면 정방향+역방향 둘 다 처리
)

print("=== nn.LSTM 파라미터 상세 ===\n")
print(f"input_size = 10:      입력 벡터 차원 (예: 단어 임베딩 크기)")
print(f"hidden_size = 20:     은닉 상태 차원 (LSTM의 기억 용량)")
print(f"num_layers = 1:       LSTM 층 수 (스택)")
print(f"batch_first = True:   입력 shape = (batch, seq_len, input_size)")
print(f"bidirectional = False: 단방향 LSTM")

# 내부 파라미터 확인
print("\n=== 내부 파라미터 (가중치) ===\n")
for name, param in lstm_layer.named_parameters():  # 모든 파라미터 순회
    print(f"  {name:25s} shape: {str(param.shape):15s} = {param.numel():5d}개 파라미터")
    # 각 파라미터의 이름, 형태, 원소 수 출력

print(f"\n  총 파라미터 수: {sum(p.numel() for p in lstm_layer.parameters()):,}개")

# 파라미터 이름 해석
print("\n=== 파라미터 이름 해석 ===\n")
print("  weight_ih_l0: 입력→은닉 가중치 (Input-Hidden), 층 0")
print("    shape = (4*hidden_size, input_size) = (80, 10)")
print("    → 4개 게이트 × hidden_size 행 × input_size 열")
print("    → [W_i, W_f, W_g, W_o] 순서로 쌓여있음 (g = cell gate)")
print()
print("  weight_hh_l0: 은닉→은닉 가중치 (Hidden-Hidden), 층 0")
print("    shape = (4*hidden_size, hidden_size) = (80, 20)")
print("    → 이전 은닉 상태 h_{t-1}에 곱하는 가중치")
print()
print("  bias_ih_l0, bias_hh_l0: 입력/은닉 편향")
print("    shape = (4*hidden_size,) = (80,)")
print("    → 최종 편향 = bias_ih + bias_hh")

# PyTorch는 W_ih와 W_hh를 분리하지만,
# 우리 구현에서는 [h, x]를 연결한 후 하나의 W를 곱했음
print("\n💡 우리 구현 vs PyTorch:")
print("  우리: W_f · [h, x] + b_f")
print("  PyTorch: W_ih · x + W_hh · h + b_ih + b_hh (수학적으로 동일!)")

In [ ]:
# ============================================================
# Part 3.1 (계속): nn.LSTM 입출력 확인
# 실제 텐서를 넣어서 입력/출력의 형태를 확인합니다
# ============================================================

# 입력 텐서 생성
batch_size = 3   # 배치 크기 (한 번에 처리할 시퀀스 수)
seq_len = 7      # 시퀀스 길이 (시점 수)
input_size = 10  # 입력 차원
hidden_size = 20  # 은닉 차원

# 랜덤 입력 텐서 생성
x = torch.randn(batch_size, seq_len, input_size)  # shape: (3, 7, 10)
print(f"입력 x shape: {x.shape}")  # 입력 형태 확인
print(f"  = (배치={batch_size}, 시퀀스길이={seq_len}, 입력차원={input_size})")

# 초기 상태 (생략하면 자동으로 0으로 초기화됨)
h0 = torch.zeros(1, batch_size, hidden_size)  # 초기 은닉 상태
c0 = torch.zeros(1, batch_size, hidden_size)  # 초기 셀 상태
# shape: (num_layers * num_directions, batch_size, hidden_size)
print(f"\n초기 h0 shape: {h0.shape} = (층수×방향수, 배치, 은닉차원)")
print(f"초기 c0 shape: {c0.shape} = (층수×방향수, 배치, 은닉차원)")

# LSTM 순전파
output, (h_n, c_n) = lstm_layer(x, (h0, c0))  # 순전파 실행

print(f"\n=== 출력 결과 ===")
print(f"output shape: {output.shape}")  # 전체 시점의 출력
print(f"  = (배치={batch_size}, 시퀀스길이={seq_len}, 은닉차원={hidden_size})")
print(f"  → 모든 시점의 은닉 상태 h_1, h_2, ..., h_T")

print(f"\nh_n shape: {h_n.shape}")  # 마지막 시점의 은닉 상태
print(f"  = (층수×방향수=1, 배치={batch_size}, 은닉차원={hidden_size})")
print(f"  → 마지막 시점의 은닉 상태만 (h_T)")

print(f"\nc_n shape: {c_n.shape}")  # 마지막 시점의 셀 상태
print(f"  = (층수×방향수=1, 배치={batch_size}, 은닉차원={hidden_size})")
print(f"  → 마지막 시점의 셀 상태만 (c_T)")

# output의 마지막 시점 = h_n 확인
print(f"\n✓ output[:, -1, :] == h_n 확인:")
print(f"  차이: {(output[:, -1, :] - h_n.squeeze(0)).abs().max().item():.10f}")
print(f"  → 거의 0 = 동일한 값! (output의 마지막 = h_n)")

In [ ]:
# ============================================================
# Part 3.1 (계속): 다층(Multi-layer) & 양방향(Bidirectional) LSTM
# 심화 옵션들의 차이를 확인합니다
# ============================================================

print("=== 다층 LSTM (num_layers=2) ===\n")

# 2층 LSTM 생성
lstm_2layers = nn.LSTM(
    input_size=10,    # 입력 차원
    hidden_size=20,   # 은닉 차원
    num_layers=2,     # 2개 층 스택
    batch_first=True,  # 배치 우선
    dropout=0.1       # 층 사이 드롭아웃 10%
)

# 초기 상태 (2층이므로 첫 번째 차원이 2)
h0_2 = torch.zeros(2, batch_size, hidden_size)  # 2층 초기 은닉 상태
c0_2 = torch.zeros(2, batch_size, hidden_size)  # 2층 초기 셀 상태

# 순전파
output_2, (h_n_2, c_n_2) = lstm_2layers(x, (h0_2, c0_2))  # 2층 LSTM 실행

print(f"입력 shape:   {x.shape}")
print(f"output shape: {output_2.shape}")
print(f"h_n shape:    {h_n_2.shape} → h_n[0]=1층 마지막, h_n[1]=2층 마지막")
print(f"c_n shape:    {c_n_2.shape}")
print(f"파라미터 수:   {sum(p.numel() for p in lstm_2layers.parameters()):,}개")

print("\n구조 설명:")
print("  층 1: 입력(10) → LSTM → 은닉(20)")
print("  층 2: 은닉(20) → LSTM → 은닉(20)  ← 층1의 출력이 층2의 입력!")
print("  드롭아웃은 층1→층2 사이에 적용 (마지막 층 출력에는 미적용)")

print("\n" + "=" * 60)
print("=== 양방향 LSTM (bidirectional=True) ===\n")

# 양방향 LSTM 생성
lstm_bidir = nn.LSTM(
    input_size=10,       # 입력 차원
    hidden_size=20,      # 은닉 차원
    num_layers=1,        # 1개 층
    batch_first=True,    # 배치 우선
    bidirectional=True   # 양방향!
)

# 초기 상태 (양방향이므로 방향 수 = 2)
h0_bi = torch.zeros(2, batch_size, hidden_size)  # 2방향 초기 은닉 상태
c0_bi = torch.zeros(2, batch_size, hidden_size)  # 2방향 초기 셀 상태

# 순전파
output_bi, (h_n_bi, c_n_bi) = lstm_bidir(x, (h0_bi, c0_bi))  # 양방향 LSTM 실행

print(f"입력 shape:   {x.shape}")
print(f"output shape: {output_bi.shape}")
print(f"  → 은닉차원이 {hidden_size}×2 = {hidden_size*2}! (정방향+역방향 연결)")
print(f"h_n shape:    {h_n_bi.shape} → h_n[0]=정방향, h_n[1]=역방향")
print(f"파라미터 수:   {sum(p.numel() for p in lstm_bidir.parameters()):,}개")

print("\n구조 설명:")
print("  정방향: x_1 → x_2 → x_3 → ... → x_T  (왼→오)")
print("  역방향: x_T → x_{T-1} → ... → x_1      (오→왼)")
print("  출력: [정방향 h_t ; 역방향 h_t] 연결 → 2H 차원")
print("  장점: 미래 문맥도 활용 가능 (기계 번역 등에 유용)")

---

## 3.2 감정 분류 예제 (Sentiment Classification)

PyTorch nn.LSTM을 사용하여 **영화 리뷰의 감정(긍정/부정)**을 분류하는 모델을 만듭니다.

간단한 예제를 위해 **직접 만든 작은 데이터셋**을 사용합니다.

In [ ]:
# ============================================================
# Part 3.2: 감정 분류 예제 데이터셋 준비
# 간단한 영어 문장들로 감정 분류 데이터를 만듭니다
# ============================================================

# 간단한 감정 분류 데이터셋 (영어 문장)
train_data = [
    # (문장, 레이블): 1 = 긍정, 0 = 부정
    ("this movie is great", 1),        # 이 영화는 훌륭하다 → 긍정
    ("i love this film", 1),           # 이 영화를 사랑한다 → 긍정
    ("excellent performance", 1),       # 뛰어난 연기 → 긍정
    ("wonderful story telling", 1),     # 훌륭한 스토리텔링 → 긍정
    ("best movie ever", 1),            # 역대 최고 영화 → 긍정
    ("really enjoyed it", 1),          # 정말 즐거웠다 → 긍정
    ("amazing acting", 1),             # 놀라운 연기 → 긍정
    ("highly recommend this", 1),      # 강력 추천 → 긍정
    ("this movie is terrible", 0),     # 이 영화는 끔찍하다 → 부정
    ("worst film ever", 0),            # 역대 최악 영화 → 부정
    ("boring and slow", 0),            # 지루하고 느리다 → 부정
    ("waste of time", 0),              # 시간 낭비 → 부정
    ("awful performance", 0),          # 끔찍한 연기 → 부정
    ("really bad movie", 0),           # 정말 나쁜 영화 → 부정
    ("disappointing story", 0),        # 실망스러운 이야기 → 부정
    ("do not watch this", 0),          # 이거 보지 마라 → 부정
]

# 어휘 사전 구축 (단어 → 인덱스 매핑)
word_to_idx = {"<PAD>": 0, "<UNK>": 1}  # 특수 토큰: 패딩과 미등록 단어
idx = 2  # 다음 인덱스 시작값

for sentence, _ in train_data:  # 모든 학습 문장에 대해
    for word in sentence.split():  # 공백으로 단어 분리
        if word not in word_to_idx:  # 새로운 단어이면
            word_to_idx[word] = idx  # 인덱스 할당
            idx += 1  # 다음 인덱스

vocab_size = len(word_to_idx)  # 어휘 사전 크기
print(f"어휘 사전 크기: {vocab_size}개 단어")  # 어휘 크기 출력
print(f"어휘 사전 예시: {dict(list(word_to_idx.items())[:10])}")  # 일부 출력

# 문장을 인덱스 시퀀스로 변환하는 함수
def encode_sentence(sentence, word_to_idx, max_len=6):
    """문장을 정수 인덱스 시퀀스로 변환
    
    Args:
        sentence: 입력 문장 (문자열)
        word_to_idx: 단어-인덱스 매핑 딕셔너리
        max_len: 최대 시퀀스 길이 (짧으면 패딩, 길면 자르기)
    Returns:
        인덱스 리스트, 길이 = max_len
    """
    tokens = sentence.split()  # 공백으로 단어 분리
    indices = [word_to_idx.get(w, 1) for w in tokens]  # 단어→인덱스 변환 (미등록=1)
    # 패딩 처리: max_len보다 짧으면 0으로 채움
    if len(indices) < max_len:  # 짧으면
        indices += [0] * (max_len - len(indices))  # 패딩 추가
    else:  # 길면
        indices = indices[:max_len]  # 잘라내기
    return indices  # 인덱스 리스트 반환

# 전체 데이터를 텐서로 변환
max_len = 6  # 최대 시퀀스 길이
X_sentences = []  # 인코딩된 문장 리스트
y_labels = []     # 레이블 리스트

for sentence, label in train_data:  # 각 학습 데이터에 대해
    encoded = encode_sentence(sentence, word_to_idx, max_len)  # 문장 인코딩
    X_sentences.append(encoded)  # 인코딩 결과 저장
    y_labels.append(label)       # 레이블 저장

X_tensor = torch.LongTensor(X_sentences)  # 정수 텐서로 변환 (인덱스이므로 Long)
y_tensor = torch.FloatTensor(y_labels)    # 실수 텐서로 변환 (BCE 손실 사용)

print(f"\n인코딩된 입력 shape: {X_tensor.shape} = (샘플수, 최대길이)")  # 입력 형태
print(f"레이블 shape: {y_tensor.shape}")  # 레이블 형태
print(f"\n인코딩 예시:")
print(f"  '{train_data[0][0]}' → {X_tensor[0].tolist()}")  # 첫 번째 예시
print(f"  '{train_data[8][0]}' → {X_tensor[8].tolist()}")  # 아홉 번째 예시

In [ ]:
# ============================================================
# Part 3.2 (계속): PyTorch LSTM 감정 분류 모델 정의
# nn.Embedding + nn.LSTM + nn.Linear으로 구성합니다
# ============================================================

class SentimentLSTM(nn.Module):
    """LSTM 기반 감정 분류 모델
    
    구조: Embedding → LSTM → Linear → Sigmoid
    
    Args:
        vocab_size: 어휘 사전 크기
        embedding_dim: 임베딩 차원 (단어를 밀집 벡터로 변환)
        hidden_dim: LSTM 은닉 차원
        output_dim: 출력 차원 (이진 분류이므로 1)
        n_layers: LSTM 층 수
        dropout: 드롭아웃 비율
    """
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers=1, dropout=0.0):
        """모델 초기화"""
        super().__init__()  # 부모 클래스 초기화
        
        # 임베딩 레이어: 정수 인덱스 → 밀집 벡터
        # 예: 단어 "great"의 인덱스 5 → [0.2, -0.1, 0.8, ...] (embedding_dim차원)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,    # 어휘 크기 (가능한 인덱스 수)
            embedding_dim=embedding_dim,   # 각 단어의 벡터 차원
            padding_idx=0                  # 패딩 토큰(0)은 항상 0벡터로 유지
        )
        
        # LSTM 레이어
        self.lstm = nn.LSTM(
            input_size=embedding_dim,   # 입력 차원 = 임베딩 차원
            hidden_size=hidden_dim,     # 은닉 차원
            num_layers=n_layers,        # 층 수
            batch_first=True,           # 배치 우선 형태
            dropout=dropout if n_layers > 1 else 0  # 1층이면 드롭아웃 불필요
        )
        
        # 출력 레이어: 은닉 상태 → 감정 점수
        self.fc = nn.Linear(hidden_dim, output_dim)  # 선형 변환: (hidden_dim,) → (output_dim,)
        
        # 시그모이드: 감정 점수를 0~1 확률로 변환
        self.sigmoid = nn.Sigmoid()  # 출력을 확률로 변환
        
    def forward(self, text):
        """순전파
        
        Args:
            text: 정수 인덱스 텐서, shape = (batch, seq_len)
        Returns:
            확률값, shape = (batch,)
        """
        # Step 1: 임베딩
        # (batch, seq_len) → (batch, seq_len, embedding_dim)
        embedded = self.embedding(text)  # 정수 인덱스를 밀집 벡터로 변환
        
        # Step 2: LSTM
        # 입력: (batch, seq_len, embedding_dim)
        # 출력: output=(batch, seq_len, hidden_dim), (h_n, c_n)
        output, (h_n, c_n) = self.lstm(embedded)  # LSTM 순전파
        # h_n: 마지막 시점의 은닉 상태 → 문장 전체의 "요약"
        # c_n: 마지막 시점의 셀 상태 → 장기 기억
        
        # Step 3: 마지막 은닉 상태로 분류
        # h_n shape: (num_layers, batch, hidden_dim) → 마지막 층만 사용
        hidden = h_n[-1]  # 마지막 층의 은닉 상태, shape: (batch, hidden_dim)
        
        # Step 4: 선형 변환 + 시그모이드
        logit = self.fc(hidden)  # (batch, hidden_dim) → (batch, 1)
        prob = self.sigmoid(logit)  # (batch, 1) → 0~1 확률
        
        return prob.squeeze(-1)  # (batch,)로 차원 축소
    
    def predict_with_states(self, text):
        """hidden state와 cell state를 함께 반환하는 예측 함수
        
        Args:
            text: 정수 인덱스 텐서
        Returns:
            prob: 확률값
            h_n: 마지막 은닉 상태
            c_n: 마지막 셀 상태
            all_output: 모든 시점의 은닉 상태
        """
        embedded = self.embedding(text)  # 임베딩 변환
        output, (h_n, c_n) = self.lstm(embedded)  # LSTM 순전파
        hidden = h_n[-1]  # 마지막 층 은닉 상태
        logit = self.fc(hidden)  # 선형 변환
        prob = self.sigmoid(logit)  # 확률 변환
        return prob.squeeze(-1), h_n, c_n, output  # 모든 상태 반환


# 모델 생성
model_pt = SentimentLSTM(
    vocab_size=vocab_size,    # 어휘 크기
    embedding_dim=16,         # 임베딩 차원 (작은 데이터셋이므로 작게)
    hidden_dim=32,            # 은닉 차원
    output_dim=1,             # 이진 분류
    n_layers=1                # 1층 LSTM
)

print("=== SentimentLSTM 모델 구조 ===\n")
print(model_pt)  # 모델 구조 출력

# 파라미터 수 확인
total_params = sum(p.numel() for p in model_pt.parameters())  # 총 파라미터 수
print(f"\n총 파라미터 수: {total_params:,}개")

# 각 레이어별 파라미터 수
for name, param in model_pt.named_parameters():  # 각 파라미터에 대해
    print(f"  {name:35s} shape: {str(param.shape):20s} = {param.numel():6,}개")

In [ ]:
# ============================================================
# Part 3.2 (계속): 모델 학습
# BCELoss(이진 교차 엔트로피)와 Adam 옵티마이저로 학습합니다
# ============================================================

# 손실 함수와 옵티마이저 설정
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss (이진 분류용)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=0.01)  # Adam 옵티마이저, 학습률 0.01

# 학습 루프
n_epochs_pt = 100  # 에폭 수
train_losses = []  # 에폭별 손실 저장
train_accs = []    # 에폭별 정확도 저장

print("=== PyTorch LSTM 감정 분류 학습 시작 ===\n")

for epoch in range(n_epochs_pt):  # 각 에폭에 대해
    model_pt.train()  # 학습 모드 설정 (드롭아웃 활성화 등)
    
    # 순전파
    predictions = model_pt(X_tensor)  # 모든 학습 데이터에 대해 예측
    loss = criterion(predictions, y_tensor)  # 손실 계산
    
    # 역전파
    optimizer.zero_grad()  # 기울기 초기화 (이전 기울기 제거)
    loss.backward()  # 역전파로 기울기 계산
    optimizer.step()  # 파라미터 업데이트
    
    # 정확도 계산
    predicted_labels = (predictions > 0.5).float()  # 0.5 기준으로 이진 분류
    accuracy = (predicted_labels == y_tensor).float().mean()  # 정확도 계산
    
    train_losses.append(loss.item())  # 손실 저장
    train_accs.append(accuracy.item())  # 정확도 저장
    
    if (epoch + 1) % 20 == 0:  # 20 에폭마다 출력
        print(f"  에폭 {epoch+1:3d}/{n_epochs_pt}: 손실 = {loss.item():.4f}, 정확도 = {accuracy.item():.1%}")

# 학습 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1행 2열

# 손실 그래프
axes[0].plot(train_losses, 'b-', linewidth=2)  # 손실 변화 그래프
axes[0].set_xlabel('에폭', fontsize=12)  # x축
axes[0].set_ylabel('BCE 손실', fontsize=12)  # y축
axes[0].set_title('학습 손실 (PyTorch LSTM)', fontsize=14)  # 제목
axes[0].grid(True, alpha=0.3)  # 격자

# 정확도 그래프
axes[1].plot(train_accs, 'r-', linewidth=2)  # 정확도 변화 그래프
axes[1].set_xlabel('에폭', fontsize=12)  # x축
axes[1].set_ylabel('정확도', fontsize=12)  # y축
axes[1].set_title('학습 정확도 (PyTorch LSTM)', fontsize=14)  # 제목
axes[1].set_ylim(0, 1.05)  # y축 범위
axes[1].grid(True, alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print(f"\n최종 정확도: {train_accs[-1]:.1%}")

In [ ]:
# ============================================================
# Part 3.2 (계속): 새로운 문장으로 감정 예측 테스트
# 학습에 사용하지 않은 문장들로 모델을 테스트합니다
# ============================================================

# 테스트 문장들
test_sentences = [
    "this is great",           # 긍정 (예상)
    "terrible movie",          # 부정 (예상)
    "really great film",       # 긍정 (예상)
    "bad and boring",          # 부정 (예상)
    "love this performance",   # 긍정 (예상)
    "worst movie ever",        # 부정 (예상)
]

print("=== 감정 예측 테스트 ===\n")

model_pt.eval()  # 평가 모드 설정 (드롭아웃 비활성화)

with torch.no_grad():  # 기울기 계산 비활성화 (추론 시에는 불필요)
    for sentence in test_sentences:  # 각 테스트 문장에 대해
        # 문장 인코딩
        encoded = encode_sentence(sentence, word_to_idx, max_len)  # 인코딩
        input_tensor = torch.LongTensor([encoded])  # 텐서 변환 (배치 차원 추가)
        
        # 예측
        prob = model_pt(input_tensor)  # 확률 예측
        sentiment = "긍정 😊" if prob.item() > 0.5 else "부정 😢"  # 감정 판정
        
        print(f"  '{sentence}'")  # 문장 출력
        print(f"    → 확률: {prob.item():.4f}, 판정: {sentiment}")  # 예측 결과
        print()

---

## 3.3 Hidden State와 Cell State의 차이

LSTM의 두 가지 상태(Hidden State, Cell State)가 실제로 어떻게 다른지 시각적으로 확인합니다.

**핵심 차이:**
- **Cell State ($c_t$)**: 장기 기억. 값의 범위가 넓고, 천천히 변함
- **Hidden State ($h_t$)**: 단기 출력. $\tanh(c_t)$에 Output Gate를 곱한 값, -1~1 범위

In [ ]:
# ============================================================
# Part 3.3: Hidden State vs Cell State 시각적 비교
# 같은 입력에 대해 두 상태의 패턴이 어떻게 다른지 확인합니다
# ============================================================

# 긍정 문장과 부정 문장 비교
sentences_to_analyze = [
    ("this movie is great", "긍정"),  # 긍정 문장
    ("this movie is terrible", "부정"),  # 부정 문장
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))  # 2행 2열

model_pt.eval()  # 평가 모드

with torch.no_grad():  # 기울기 계산 비활성화
    for row, (sentence, label) in enumerate(sentences_to_analyze):  # 각 문장에 대해
        # 문장 인코딩 및 예측
        encoded = encode_sentence(sentence, word_to_idx, max_len)  # 인코딩
        input_tensor = torch.LongTensor([encoded])  # 텐서 변환
        
        # hidden state와 cell state를 함께 얻기
        prob, h_n, c_n, all_output = model_pt.predict_with_states(input_tensor)  # 상태 포함 예측
        
        # 단어별 토큰 준비
        words = sentence.split()  # 단어 분리
        words += ["<PAD>"] * (max_len - len(words))  # 패딩 추가
        
        # Hidden State 히트맵 (모든 시점)
        hidden_vals = all_output[0].numpy()  # shape: (seq_len, hidden_dim)
        im1 = axes[row, 0].imshow(hidden_vals.T, aspect='auto', cmap='RdBu_r')  # 히트맵
        axes[row, 0].set_title(f'Hidden State (h_t) - {label}: "{sentence}"', fontsize=11)  # 제목
        axes[row, 0].set_xlabel('시점 (단어)', fontsize=10)  # x축
        axes[row, 0].set_ylabel('은닉 유닛', fontsize=10)  # y축
        axes[row, 0].set_xticks(range(len(words)))  # x축 눈금
        axes[row, 0].set_xticklabels(words, rotation=45, ha='right')  # 단어 레이블
        plt.colorbar(im1, ax=axes[row, 0])  # 컬러바
        
        # Cell State는 LSTM 내부에서 직접 접근해야 하므로,
        # 수동으로 각 시점의 cell state를 추출합니다
        
        # LSTM을 수동으로 한 시점씩 실행하여 Cell State 추출
        embedded = model_pt.embedding(input_tensor)  # 임베딩: (1, seq_len, emb_dim)
        h_manual = torch.zeros(1, 1, 32)  # 초기 은닉 상태 (1층, 배치1, 32차원)
        c_manual = torch.zeros(1, 1, 32)  # 초기 셀 상태
        cell_states_list = []  # 셀 상태 저장용
        
        for t in range(max_len):  # 각 시점에 대해
            # 한 시점씩 LSTM 실행
            out_t, (h_manual, c_manual) = model_pt.lstm(
                embedded[:, t:t+1, :], (h_manual, c_manual)  # 한 시점 입력
            )
            cell_states_list.append(c_manual[0, 0].numpy().copy())  # 셀 상태 저장
        
        cell_vals = np.array(cell_states_list)  # shape: (seq_len, hidden_dim)
        im2 = axes[row, 1].imshow(cell_vals.T, aspect='auto', cmap='RdBu_r')  # 히트맵
        axes[row, 1].set_title(f'Cell State (c_t) - {label}: "{sentence}"', fontsize=11)  # 제목
        axes[row, 1].set_xlabel('시점 (단어)', fontsize=10)  # x축
        axes[row, 1].set_ylabel('셀 유닛', fontsize=10)  # y축
        axes[row, 1].set_xticks(range(len(words)))  # x축 눈금
        axes[row, 1].set_xticklabels(words, rotation=45, ha='right')  # 단어 레이블
        plt.colorbar(im2, ax=axes[row, 1])  # 컬러바
        
        print(f"'{sentence}' ({label})")
        print(f"  예측 확률: {prob.item():.4f}")
        print(f"  Hidden State 범위: [{hidden_vals.min():.4f}, {hidden_vals.max():.4f}]")
        print(f"  Cell State 범위:   [{cell_vals.min():.4f}, {cell_vals.max():.4f}]")
        print()

plt.suptitle('Hidden State vs Cell State 비교', fontsize=14, y=1.02)  # 전체 제목
plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print("🔑 관찰:")
print("  1. Cell State의 값 범위가 Hidden State보다 넓을 수 있음 (tanh 제한 없음)")
print("  2. Hidden State는 Output Gate로 필터링된 '공개 정보'")
print("  3. Cell State는 LSTM 내부에서만 순환하는 '비공개 기억'")
print("  4. 긍정/부정 문장에서 각 유닛의 활성화 패턴이 다름 → 감정 구분 학습!")

---

# Part 4: 정리

---

## 4.1 LSTM의 남은 한계

LSTM은 RNN의 기울기 소실 문제를 크게 완화했지만, 여전히 **근본적인 한계**가 있습니다:

### 1. 순차 처리의 병목 (Sequential Processing Bottleneck)

```
시점 1 → 시점 2 → 시점 3 → ... → 시점 T
(이전 시점이 끝나야 다음 시점 처리 가능)
```

- LSTM은 시점을 **하나씩 순서대로** 처리해야 합니다
- GPU의 **병렬 처리 능력**을 충분히 활용할 수 없습니다
- 시퀀스 길이가 길어지면 **학습/추론 시간이 선형적으로 증가**
- 대규모 데이터셋 학습에 **비효율적**

### 2. 먼 거리 의존성의 한계 (Long-Range Dependencies)

- LSTM이 기울기 소실을 완화했지만, **완전히 해결한 것은 아닙니다**
- Forget Gate를 통한 곱셈이 반복되면 여전히 **정보 손실** 발생
- 수백~수천 시점 이상의 아주 긴 시퀀스에서는 성능 저하
- 실험적으로 약 **200~300 시점** 이상에서 어려움

### 3. 고정 크기 은닉 상태 (Fixed-size Hidden State)

- 전체 시퀀스의 정보를 **하나의 고정 크기 벡터($h_T$)**에 압축
- 시퀀스가 길어질수록 정보 **병목(bottleneck)** 발생
- 긴 문서의 모든 세부사항을 기억하기 어려움

---

## 4.2 Attention 메커니즘으로의 발전 동기

위 한계들을 극복하기 위해 **Attention 메커니즘**이 등장합니다:

| LSTM의 한계 | Attention의 해결 |
|---|---|
| 순차 처리 → 느림 | **모든 시점을 동시에** 참조 (병렬화 가능) |
| 먼 거리 의존성 한계 | 어떤 시점이든 **직접 연결** (거리 무관) |
| 고정 크기 은닉 상태 | **모든 시점의 정보**를 가중 합산 |

Attention의 핵심 아이디어:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

> 💡 **다음 단계**: Attention 메커니즘 → Self-Attention → **Transformer** (2017)
> → BERT, GPT 등 현대 언어 모델의 기반!

---

## 4.3 핵심 수식 요약표

| 수식 | 이름 | 역할 | 출력 범위 |
|------|------|------|-----------|
| $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | Forget Gate | 이전 기억 중 **잊을 비율** | $(0, 1)$ |
| $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | Input Gate | 새 정보를 **기억할 비율** | $(0, 1)$ |
| $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | Candidate | **새로 만든 정보** | $(-1, 1)$ |
| $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | Cell Update | 기억 **갱신** (핵심!) | 제한 없음 |
| $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | Output Gate | 셀에서 **출력할 비율** | $(0, 1)$ |
| $h_t = o_t \odot \tanh(c_t)$ | Hidden State | **최종 출력** | $(-1, 1)$ |

### 가중치 차원 요약

| 파라미터 | 차원 | 개수 |
|----------|------|------|
| $W_f, W_i, W_c, W_o$ (각각) | $(H, H+d)$ | $H \times (H + d)$ |
| $b_f, b_i, b_c, b_o$ (각각) | $(H,)$ | $H$ |
| **총 파라미터** | | $4H(H + d) + 4H = 4H(H + d + 1)$ |

### 기억할 핵심 포인트

1. **Cell State의 덧셈 구조** ($c_t = ... + ...$)가 기울기 소실을 완화하는 핵심
2. **시그모이드 = 게이트** (0~1 = 얼마나 통과시킬지), **tanh = 값** (-1~1 = 어떤 정보인지)
3. **$b_f$를 1로 초기화**하면 학습 초기에 기울기가 잘 흐름
4. Hidden State는 "말한 것", Cell State는 "머릿속 생각"

---

## 참고 자료

- Hochreiter & Schmidhuber (1997). "Long Short-Term Memory"
- Gers et al. (2000). "Learning to Forget: Continual Prediction with LSTM"
- Cho et al. (2014). "Learning Phrase Representations using RNN Encoder-Decoder" (GRU)
- Jozefowicz et al. (2015). "An Empirical Exploration of Recurrent Network Architectures"
- Olah, C. (2015). "Understanding LSTM Networks" (colah.github.io)
- Greff et al. (2017). "LSTM: A Search Space Odyssey"

> 다음 챕터에서는 **Attention 메커니즘**을 배우고, LSTM의 한계를 어떻게 극복하는지 알아봅니다!